# HS forecast model: predict IHS4 for the next 6 months

**Goal.** For every patient, using only what is known on **31 Dec 2025**, predict the average underlying IHS4 over 1 Jan - 30 Jun 2026 (`avg_ihs4_next6`).
Inputs: demography, practitioner pairing, surgeries, treatments, visit time series, and (for some patients) RNAseq and image embeddings.

### How findings are labelled
Every finding in the model is tagged:

| Label | Meaning | What we do with it |
|---|---|---|
| **CLINICAL** | Reflects the disease or treatment. Would appear in a real population, should carry to test_2. | Safe to use. |
| **ARTEFACT** | Comes from how data were recorded / collected / built. May predict here, but fragile. | Do not rely on it unless an ablation says it pays *and* it survives held-out-group validation. Always flag it. |
| **UNDETERMINED** | A pattern is visible but the data cannot yet separate the two explanations. | Say so, and write down what evidence would settle it. |

## 0. Setup and the validation ledger
Everything that can be switched on/off lives in `CFG`. The **ledger** (`check(...)`) records one PASS/FAIL line per validation. 
`EV` is a dictionary where numbers computed in each step are stored; the final *findings table* reads from it so the evidence is always the real number.

In [2]:
# packages

# enabler
import os, warnings, time, json; warnings.filterwarnings("ignore")
import urllib.request
import urllib.error

# standard
import numpy as np, pandas as pd, seaborn as sns
import matplotlib.pyplot as plt

# model
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor as HGBR, HistGradientBoostingClassifier as HGBC
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV


from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.mixture import GaussianMixture
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 100

# colab output
from google.colab import files       

In [3]:
# Train data
DATA_DIR = "https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data"        # folder with the TRAIN csv files
END = pd.Timestamp("2025-12-31")                       # nothing after this date may be used
CFG = dict(use_practitioner=False,   # artefact family; decided by ablation in step 6
           use_hidden_flare=True,    # decided by ablation in step 6
           use_response_types=False) # decided by ablation gate in step 9
LEDGER, EV = [], {}
def check(step, name, ok, detail="", warn=False):
    # warn=True: a known limitation that is reported as WARN instead of FAIL when it does not hold
    status = "PASS" if ok else ("WARN" if warn else "FAIL")
    LEDGER.append(dict(step=step, check=name, passed=bool(ok), status=status, detail=str(detail)))
    print(status, "|", step, "|", name, "|", detail)
mae = lambda y, p: float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))
def paired_gain(e0, e1, n=1000, seed=0):
    # gain = mean(error_old - error_new) with a paired bootstrap CI over patients. Positive = new is better.
    d = np.asarray(e0) - np.asarray(e1); r = np.random.default_rng(seed)
    m = np.concatenate([d[r.integers(0, len(d), (100, len(d)))].mean(1) for _ in range(n // 100)])
    return dict(gain=float(d.mean()), lo=float(np.percentile(m, 2.5)), hi=float(np.percentile(m, 97.5)))
fmt = lambda g: f"{g['gain']:+.4f} (95% CI {g['lo']:+.4f} .. {g['hi']:+.4f})"

## 1. Load the data and check it (validation V1)
Every later number depends on the tables being what we think they are. Leakage often begins here: a date after the cutoff, a duplicated patient, a join silently drops people.

Checks: unique ids, all dates `<= 2025-12-31`, at most one open biologic line per patient, every table's patients exist in demographics, targets are positive and complete.

In [5]:
# function helpers -- general

# path helper 
def _src(d, name):
    """GitHub blob link -> raw link. Local folder -> normal path."""
    d = str(d).rstrip("/")
    if d.startswith("http"):
        d = (d.replace("https://github.com/", "https://raw.githubusercontent.com/")
              .replace("/blob/", "/"))
    return f"{d}/{name}"

def _read_visits(d, **kw):
    """visits.csv if present, else merge visits1.csv + visits2.csv."""
    try:
        return pd.read_csv(_src(d, "visits.csv"), **kw)
    except (FileNotFoundError, OSError):          # HTTPError is an OSError
        v1 = pd.read_csv(_src(d, "visits1.csv"), **kw)
        v2 = pd.read_csv(_src(d, "visits2.csv"), **kw)
        assert list(v1.columns) == list(v2.columns), "visits1/visits2 columns differ"
        out = pd.concat([v1, v2], ignore_index=True).drop_duplicates()
        print(f"visits merged: {len(v1)} + {len(v2)} -> {len(out)}")
        return out

def load_tables(d):
    # Missing patients in a table are normal (e.g. 398 patients have no visits).
    dem = pd.read_csv(_src(d, "demographics.csv"), parse_dates=["symptom_onset_date", "diagnosis_date"])
    asg = pd.read_csv(_src(d, "patient_assignment.csv"))
    trt = pd.read_csv(_src(d, "treatments.csv"), parse_dates=["start_date", "end_date"])
    sur = pd.read_csv(_src(d, "surgeries.csv"), parse_dates=["procedure_date"])
    vis = (_read_visits(d, parse_dates=["visit_date"])
           .sort_values(["patient_id", "visit_date"]).reset_index(drop=True))
    return dict(dem=dem, asg=asg, trt=trt, sur=sur, vis=vis)

def load_targets(d):
    return pd.read_csv(_src(d, "targets.csv")).set_index("patient_id").avg_ihs4_next6

# checks
def check_tables(T, step="V1 data", cutoff=END):
    dem, asg, trt, sur, vis = T["dem"], T["asg"], T["trt"], T["sur"], T["vis"]
    check(step, "patient_id unique in demographics", dem.patient_id.is_unique)
    check(step, "patient_id unique in assignment", asg.patient_id.is_unique)
    mx = max(vis.visit_date.max(), sur.procedure_date.max(),
             trt.start_date.max(), dem.diagnosis_date.max())
    check(step, f"no start/event date after {cutoff.date()}", mx <= cutoff, f"latest = {mx.date()}")
    check(step, "treatment end dates never after cutoff", (trt.end_date.dropna() <= cutoff).all())
    opn = trt[(trt.treatment_class == "Biologic") & trt.end_date.isna()].groupby("patient_id").size()
    check(step, "at most one open biologic line per patient", (opn <= 1).all(),
          f"{int((opn > 0).sum())} patients on a biologic")
    known = set(dem.patient_id)
    for nm in ("trt", "sur", "vis"):
        check(step, f"{nm}: every patient is in demographics", set(T[nm].patient_id) <= known)
    check(step, "IHS4 scores non-negative", (vis.ihs4_score.dropna() >= 0).all(),
          f"max score {vis.ihs4_score.max():.0f}")

In [6]:
# train data set load and completeness check
T = load_tables(DATA_DIR)
y = load_targets(DATA_DIR)

check_tables(T)
check("V1 data", "targets: complete, positive, all patients in demographics",
      y.notna().all() and (y > 0).all() and set(y.index) <= set(T["dem"].patient_id),
      f"{len(y)} train patients")

print({k: v.shape for k, v in T.items()})
cov = pd.Series({
    "visits": T["vis"].patient_id.nunique(),
    "surgeries": T["sur"].patient_id.nunique(),
    "antibiotic/biologic rows": T["trt"].patient_id.nunique(),
}) / T["dem"].shape[0]
print("share of patients present in each table:\n", cov.round(3).to_string())

visits merged: 425363 + 284327 -> 709690
PASS | V1 data | patient_id unique in demographics | 
PASS | V1 data | patient_id unique in assignment | 
PASS | V1 data | no start/event date after 2025-12-31 | latest = 2025-12-31
PASS | V1 data | treatment end dates never after cutoff | 
PASS | V1 data | at most one open biologic line per patient | 8548 patients on a biologic
PASS | V1 data | trt: every patient is in demographics | 
PASS | V1 data | sur: every patient is in demographics | 
PASS | V1 data | vis: every patient is in demographics | 
PASS | V1 data | IHS4 scores non-negative | max score 45
PASS | V1 data | targets: complete, positive, all patients in demographics | 21000 train patients
{'dem': (21000, 12), 'asg': (21000, 2), 'trt': (203812, 8), 'sur': (60036, 8), 'vis': (709690, 6)}
share of patients present in each table:
 visits                      0.981
surgeries                   0.587
antibiotic/biologic rows    0.966


In [7]:
# check data content
END = pd.Timestamp("2025-12-31")

# plain string, NOT Path()
DATA_DIR = "https://github.com/FairzanIsLearning/Chemometric-learn/blob/main/Mixed/HS_Model/Raw_data"

if "check" not in globals():
    def check(step, name, cond, detail=""):
        print(("PASS" if bool(cond) else "FAIL"), f"[{step}] {name}", detail)

check_tables(T)
check("V1 data", "targets: complete, positive, all patients in demographics",
      y.notna().all() and (y > 0).all() and set(y.index) <= set(T["dem"].patient_id), f"{len(y)} train patients")
print({k: v.shape for k, v in T.items()})
cov = pd.Series({"visits": T["vis"].patient_id.nunique(),
                 "surgeries": T["sur"].patient_id.nunique(),
                 "antibiotic/biologic rows": T["trt"].patient_id.nunique()}) / T["dem"].shape[0]
print("share of patients present in each table:\n", cov.round(3).to_string())

PASS | V1 data | patient_id unique in demographics | 
PASS | V1 data | patient_id unique in assignment | 
PASS | V1 data | no start/event date after 2025-12-31 | latest = 2025-12-31
PASS | V1 data | treatment end dates never after cutoff | 
PASS | V1 data | at most one open biologic line per patient | 8548 patients on a biologic
PASS | V1 data | trt: every patient is in demographics | 
PASS | V1 data | sur: every patient is in demographics | 
PASS | V1 data | vis: every patient is in demographics | 
PASS | V1 data | IHS4 scores non-negative | max score 45
PASS | V1 data | targets: complete, positive, all patients in demographics | 21000 train patients
{'dem': (21000, 12), 'asg': (21000, 2), 'trt': (203812, 8), 'sur': (60036, 8), 'vis': (709690, 6)}
share of patients present in each table:
 visits                      0.981
surgeries                   0.587
antibiotic/biologic rows    0.966


Missing data is not random by default: look at how visits are *recorded*
* **Scoring**: only some visits carry a score. Does the share scored depend on the *practitioner* more than chance allows? (If yes, that is a recording habit.)
* **Visit type**: flare and follow-up visits score higher than background/monitoring visits. Is that a real difference in disease activity at that moment, or does it come from *who gets sampled when*?

In [ ]:
# analyse the visit-related data
vis = T["vis"]            # visits.csv already carries the practitioner of each visit
print(vis.groupby("visit_type").ihs4_score.agg(n="size", mean="mean", share_scored=lambda s: s.notna().mean()).round(3).to_string())

# (a) practitioner scoring habit: observed spread of "share scored" vs what pure chance (binomial) gives
g = vis.groupby("practitioner_id").ihs4_score.agg(n="size", rate=lambda s: s.notna().mean())
g = g[g.n >= 50]; p0 = vis.ihs4_score.notna().mean()
z = (g.rate - p0) / np.sqrt(p0 * (1 - p0) / g.n)
EV["score_rate_sd"] = float(g.rate.std()); EV["score_rate_z_sd"] = float(z.std())   # z_sd ~ 1 means pure chance; much bigger means real habit
print(f"practitioners with >=50 visits: {len(g)}; sd of scoring rate = {g.rate.std():.3f}; sd of z-scores = {z.std():.2f} (chance would give ~1.0)")

# (b) within-patient visit-type effect: remove each patient's own mean, then compare types
s = vis[vis.ihs4_score.notna()].copy()
s["dev"] = s.ihs4_score - s.groupby("patient_id").ihs4_score.transform("mean")
w = s.groupby("visit_type").dev.agg(["mean", "size"]).round(3)
EV["within_patient_flare_gap"] = float(s[s.visit_type == "flare_reactive"].dev.mean() - s[s.visit_type == "background"].dev.mean())
print("\nscore minus the patient's own mean, by visit type (within-patient):\n", w.to_string())

                           n    mean  share_scored
visit_type                                        
background             84929   8.979         0.584
flare_reactive        153136  12.784         0.663
follow_up             355844  13.778         0.676
treatment_monitoring  115781   9.136         0.557
practitioners with >=50 visits: 2031; sd of scoring rate = 0.171; sd of z-scores = 6.44 (chance would give ~1.0)

score minus the patient's own mean, by visit type (within-patient):
                        mean    size
visit_type                         
background           -1.950   49583
flare_reactive        0.785  101597
follow_up             0.718  240686
treatment_monitoring -2.416   64540


**What this tells us (labels used later):**
* The spread of practitioner scoring rates is far larger than chance (z-score sd printed above). That is a **recording habit**, not biology -> **ARTEFACT** (finding F5).
* Inside the same patient, scores at flare visits are higher than at background visits. So the *score itself* tracks a real flare (**CLINICAL**, F4a). But a patient who is mostly seen at flare visits has a raw mean score that overstates their usual level, because visits are triggered by flares -> sampling **ARTEFACT** for "carry the recent scores forward" (F4b). This is why the model gets visit-type and flare-share features rather than one raw average.

# 2 - identify feature
**Every feature for a cutoff date `c` is computed from tables that were first cut at `c`.** The function `as_of(T, c)` removes every row dated after `c`, and blanks biologic end dates after `c` (on date `c` that line was still running; knowing when it later stopped would be a look into the future).
`build(T, c)` only ever sees the cut tables, so look-ahead is impossible *by construction*, and we then **test** that claim (validation V2).

Calling `build` with an earlier `c` is also what makes the time-forward backtest possible later.

| Family | Examples | Why |
|---|---|---|
| hurley | baseline Hurley stage | dominant axis of severity |
| demographics | sex, age, BMI, smoking, ethnicity, prevalent flag | small real effects; prevalent cohort has placeholder dates, so its age-at-dx / years-since-dx are set to missing |
| antibiotics, surgeries, biologic | counts in windows, days since last, on-drug, line, days on drug | treatment and activity history |
| visit_scores | mean/max/min per window, last 3, exponentially decayed means, per-visit-type means, on-drug level | recent level of disease, with noise averaged out |
| visit_behaviour | visits per window, flare share, share scored, gaps | *when* a patient comes is informative (informative visiting) |
| hidden_flare | flares that left a trace (antibiotic course, surgery) but no scored visit | the target includes flares nobody scored |
| practitioner | site scoring rate and mean score | documentation habits (artefact family, off by default) |

In [9]:
# function helpers -- general

# take data from cut off date
def as_of(T, c):
    """Return copies of every table cut at date c. Nothing after c survives.
    Biologic end dates after c are blanked: on date c the line was still running."""
    c = pd.Timestamp(c)
    trt = T["trt"][T["trt"].start_date <= c].copy()
    trt.loc[trt.end_date > c, "end_date"] = pd.NaT
    return dict(dem=T["dem"][T["dem"].diagnosis_date <= c].copy(), asg=T["asg"],
                trt=trt, sur=T["sur"][T["sur"].procedure_date <= c].copy(),
                vis=T["vis"][T["vis"].visit_date <= c].copy())

# regrouping per patient_id
def _cnt(df, col, c, days, name):
    m = (df[col] <= c) & (df[col] > c - pd.Timedelta(days=days))
    return df[m].groupby("patient_id").size().rename(name)

In [10]:
# functions helper -- analysis

# identify IHS4 signal with the same patient group7u=
def fit_uplift(Tc, c):
    """How many IHS4 points higher is a score taken during an antibiotic course (first 14 days)
    than the same patient's other scores?  Estimated per Hurley stage; no target used."""
    v = Tc["vis"]; v = v[v.ihs4_score.notna()][["patient_id","visit_date","ihs4_score"]]
    a = Tc["trt"][Tc["trt"].treatment_class == "Antibiotic"][["patient_id","start_date"]].sort_values("start_date")
    v = v.sort_values("visit_date")
    m = pd.merge_asof(v, a.rename(columns={"start_date":"visit_date"}).assign(s=lambda d: d.visit_date),
                      on="visit_date", by="patient_id", direction="backward")
    m["in_course"] = (m.visit_date - m.s).dt.days.lt(14)
    g = m.groupby(["patient_id","in_course"]).ihs4_score.mean().unstack()
    g = g.dropna()
    g["d"] = g[True] - g[False]
    h = Tc["dem"].set_index("patient_id").hurley_stage_baseline.astype(str).str.extract(r"(\d)")[0].astype(float)
    u = g.d.groupby(h.reindex(g.index)).mean()
    return u.to_dict()

# identify  hidden flares
def hidden_flare_feats(Tc, c, f, uplift):
    """Flares that left a trace (antibiotic course or surgery) but no scored visit around them."""
    c = pd.Timestamp(c)
    sc = Tc["vis"][Tc["vis"].ihs4_score.notna()][["patient_id","visit_date"]].sort_values("visit_date")
    out = {}
    for kind, tab, col, lo, hi in [("abx", Tc["trt"][Tc["trt"].treatment_class=="Antibiotic"], "start_date", 7, 21),
                                   ("sur", Tc["sur"], "procedure_date", 14, 14)]:
        e = tab[tab[col] > c - pd.Timedelta(days=365)][["patient_id", col]].copy()
        e = e[e[col] <= c - pd.Timedelta(days=hi)]      # window must be fully observable (right-censoring guard)
        lows = e[col] - pd.Timedelta(days=lo); highs = e[col] + pd.Timedelta(days=hi)
        s_ = sc.visit_date.values
        # nearest scored visit per patient via merge_asof on the window start
        e = e.assign(w0=lows, w1=highs).sort_values("w0")
        nxt = pd.merge_asof(e, sc.rename(columns={"visit_date":"nv"}), left_on="w0", right_on="nv", by="patient_id", direction="forward")
        e["seen"] = (nxt.nv.values <= nxt.w1.values)
        e = e.sort_index()
        e["age"] = (c - e[col]).dt.days
        for d in (180, 365):
            x = e[e.age < d]
            out[f"hid_{kind}_{d}"] = x[~x.seen].groupby("patient_id").size()
            out[f"seen_{kind}_{d}"] = x[x.seen].groupby("patient_id").size()
    h = pd.DataFrame(out).reindex(f.index).fillna(0)
    h["flare_cover_abx_365"] = h.seen_abx_365 / (h.seen_abx_365 + h.hid_abx_365).replace(0, np.nan)
    up = f.hurley.map(uplift)
    h["hid_burden_365"] = h.hid_abx_365 * 14 / 365 * up      # expected unseen IHS4 points, spread over the year
    h["hid_burden_180"] = h.hid_abx_180 * 14 / 180 * up
    return h.drop(columns=["seen_abx_180","seen_sur_180"])

# feature families (used for ablations and audits)
def families(cols):
    fam = {}
    for k in cols:
        if k in ("p_score_rate","p_mean","prac_num"): g = "practitioner"
        elif k.startswith("hid_") or k.startswith("seen_") or k == "flare_cover_abx_365": g = "hidden_flare"
        elif k in ("v_n_all","v_days_since","v_first_days","v_rate_trend","v_gap_med","v_gap_max_365") or k.startswith(("v_n_","v_flare_share","v_scored")): g = "visit_behaviour"
        elif k.startswith("s_"): g = "visit_scores"
        elif k.startswith(("abx_",)): g = "antibiotics"
        elif k.startswith("surg"): g = "surgeries"
        elif k in ("on_bio","cur_line","days_on_bio","bio_drug","n_lines","days_since_bio_stop","days_dx_to_bio"): g = "biologic"
        elif k in ("hurley",): g = "hurley"
        else: g = "demographics"
        fam[k] = g
    return pd.Series(fam)

In [11]:
# full feature builder
def build(Tfull, c, uplift=None, ids=None, hidden=True):
    """All features as the clinic knew them on date c."""
    c = pd.Timestamp(c); Tc = as_of(Tfull, c)
    dem, asg, trt, sur, vis = Tc["dem"], Tc["asg"], Tc["trt"], Tc["sur"], Tc["vis"]
    if ids is None: ids = pd.Index(dem.patient_id)
    ids = pd.Index(ids, name="patient_id")      # index NAME matters for pandas alignment later
    f = dem.set_index("patient_id").reindex(ids).copy()
    f["prevalent"] = (f.diagnosis_date == pd.Timestamp("2016-01-02")).astype(float)
    prev = f.prevalent == 1
    f["age"] = c.year - f.birth_year
    f["yrs_since_dx"] = ((c - f.diagnosis_date).dt.days / 365.25).where(~prev)       # prevalent dx date is a placeholder
    f["age_dx"] = f.age_at_diagnosis.where(~prev); f["delay"] = f.diagnostic_delay_years.where(~prev)
    f["sex"] = (f.sex.astype(str).str[0] == "M").astype(float)
    for k in ["ethnicity","smoking_status","family_history_hs"]:
        f[k] = f[k].astype("category").cat.codes.astype(float).replace(-1, np.nan)
    f["hurley"] = f.hurley_stage_baseline.astype(str).str.extract(r"(\d)")[0].astype(float)
    f = f.rename(columns={"bmi_at_baseline":"bmi"})
    f = f.drop(columns=["birth_year","symptom_onset_date","hurley_stage_baseline","age_at_diagnosis","diagnostic_delay_years"])
    dx = f.diagnosis_date; f = f.drop(columns="diagnosis_date")
    # ---- antibiotics
    a = trt[trt.treatment_class == "Antibiotic"]
    for d in (90, 180, 365): f = f.join(_cnt(a, "start_date", c, d, f"abx_{d}d"))
    f = f.join(a.groupby("patient_id").start_date.agg(abx_all="size", abx_last="max"))
    f["abx_since"] = (c - f.abx_last).dt.days; f = f.drop(columns="abx_last")
    # ---- biologics
    b = trt[trt.treatment_class == "Biologic"].copy(); b["open"] = b.end_date.isna()
    cur = b[b.open].sort_values("start_date").groupby("patient_id").tail(1).set_index("patient_id")
    f["on_bio"] = f.index.isin(cur.index).astype(float)
    f["cur_line"] = cur.treatment_line.reindex(f.index)
    f["days_on_bio"] = (c - cur.start_date).dt.days.reindex(f.index)
    f["bio_drug"] = cur.drug_name.astype("category").cat.codes.reindex(f.index).astype(float)
    f["n_lines"] = b.groupby("patient_id").size().reindex(f.index).fillna(0)
    f["days_since_bio_stop"] = (c - b[~b.open].groupby("patient_id").end_date.max()).dt.days.reindex(f.index)
    f["days_dx_to_bio"] = (b.groupby("patient_id").start_date.min() - dx.set_axis(f.index)).dt.days.reindex(f.index).where(~prev)
    # ---- surgeries
    for d in (180, 365, 730): f = f.join(_cnt(sur, "procedure_date", c, d, f"surg_{d}d"))
    f = f.join(sur.groupby("patient_id").procedure_date.agg(surg_all="size", surg_last="max"))
    f["surg_since"] = (c - f.surg_last).dt.days; f = f.drop(columns="surg_last")
    # ---- visits
    v = vis.copy(); v["age_d"] = (c - v.visit_date).dt.days
    v["flare"] = (v.visit_type == "flare_reactive").astype(int)
    sc = v[v.ihs4_score.notna()].copy()
    f["v_n_all"] = v.groupby("patient_id").size()
    f["v_days_since"] = v.groupby("patient_id").age_d.min()
    f["v_first_days"] = v.groupby("patient_id").age_d.max()
    for w in (30, 90, 180, 365):
        vw = v[v.age_d < w]; sw = sc[sc.age_d < w]
        gv = vw.groupby("patient_id"); gs = sw.groupby("patient_id").ihs4_score
        f[f"v_n_{w}"] = gv.size(); f[f"v_flare_share_{w}"] = gv.flare.mean()
        f[f"v_scored_{w}"] = vw.assign(s=vw.ihs4_score.notna()).groupby("patient_id").s.mean()
        f[f"s_mean_{w}"] = gs.mean(); f[f"s_max_{w}"] = gs.max(); f[f"s_min_{w}"] = gs.min()
        f[f"s_nonflare_mean_{w}"] = sw[sw.flare == 0].groupby("patient_id").ihs4_score.mean()
    for ty in ["background","follow_up","treatment_monitoring","flare_reactive"]:
        x = sc[(sc.visit_type == ty) & (sc.age_d < 365)]
        f[f"s_{ty}_365"] = x.groupby("patient_id").ihs4_score.mean()
    r = sc.groupby("patient_id").cumcount(ascending=False)
    for k in (0, 1, 2): f[f"s_last{k}"] = sc[r == k].set_index("patient_id").ihs4_score
    f["s_last0_age"] = sc.groupby("patient_id").age_d.min()
    f["s_last0_flare"] = sc[r == 0].set_index("patient_id").flare
    for tau in (45, 120, 365):
        w_ = np.exp(-sc.age_d / tau); g = pd.DataFrame({"w": w_, "ws": w_ * sc.ihs4_score, "pid": sc.patient_id}).groupby("pid").sum()
        f[f"s_ewm_{tau}"] = g.ws / g.w; f[f"s_ewm_w_{tau}"] = g.w
    nf = sc[sc.flare == 0]; w_ = np.exp(-nf.age_d / 120)
    g = pd.DataFrame({"w": w_, "ws": w_ * nf.ihs4_score, "pid": nf.patient_id}).groupby("pid").sum()
    f["s_nonflare_ewm_120"] = g.ws / g.w; f["s_nonflare_ewm_w_120"] = g.w
    f["s_all_mean"] = sc.groupby("patient_id").ihs4_score.mean()
    f["s_trend"] = f.s_mean_180 - sc[(sc.age_d >= 180) & (sc.age_d < 365)].groupby("patient_id").ihs4_score.mean()
    # on-drug level once the drug has had time to work (>=90 days after the current line started)
    if len(cur):
        st = cur.start_date.rename("t0"); x = sc.join(st, on="patient_id", how="inner")
        x = x[(x.visit_date - x.t0).dt.days >= 90]
        f["s_ondrug_mean"] = x.groupby("patient_id").ihs4_score.mean()
    else: f["s_ondrug_mean"] = np.nan
    f["v_rate_trend"] = f.v_n_180 / (f.v_n_365 - f.v_n_180).replace(0, np.nan)
    gap = v.groupby("patient_id").visit_date.diff().dt.days
    f["v_gap_med"] = gap.groupby(v.patient_id).median()
    f["v_gap_max_365"] = gap[v.age_d < 365].groupby(v.patient_id[v.age_d < 365]).max()
    # fill structural zeros
    zc = [k for k in f.columns if k.startswith(("abx_", "surg_", "v_n_"))]
    f[zc] = f[zc].fillna(0)
    # ---- practitioner features (artefact family; kept separate so they can be switched off)
    pr = asg.set_index("patient_id").practitioner_id.reindex(f.index)
    f["prac_num"] = pr.str.extract(r"(\d+)")[0].astype(float)
    pv = v.merge(asg, on="patient_id", suffixes=("", "_a"))
    ps = pv.groupby("practitioner_id_a").ihs4_score.agg(p_score_rate=lambda x: x.notna().mean(), p_mean="mean")
    f = f.join(pr.rename("_pid")).join(ps, on="_pid").drop(columns="_pid")
    if hidden:
        if uplift is None: uplift = fit_uplift(Tc, c)
        f = f.join(hidden_flare_feats(Tc, c, f, uplift))
    f.attrs["uplift"] = uplift
    return f

In [12]:
# build the training feature and test for leakage

t0 = time.time()
f_all = build(T, END)                                   # features for the 21,000 train patients (cutoff = 31 Dec 2025)
UPLIFT = f_all.attrs["uplift"]
X = f_all.loc[y.index]; fam = families(X.columns)
print(f"features: {X.shape}  built in {time.time()-t0:.1f}s; uplift per Hurley = { {k: round(v,2) for k,v in UPLIFT.items()} }")
print(fam.value_counts().to_string())

# test 1: future corruption
def corrupt_future(T, c):
    c = pd.Timestamp(c); B = {k: v.copy() for k, v in T.items()}
    fut = B["vis"].visit_date > c
    B["vis"].loc[fut, "ihs4_score"] = B["vis"].loc[fut, "ihs4_score"] + 50
    B["vis"] = pd.concat([B["vis"], B["vis"][fut].assign(visit_date=lambda d: d.visit_date + pd.Timedelta(days=400))], ignore_index=True)
    B["sur"] = B["sur"][B["sur"].procedure_date <= c]
    m = (B["trt"].treatment_class == "Biologic") & (B["trt"].end_date > c)
    B["trt"].loc[m, "end_date"] = B["trt"].loc[m, "end_date"] + pd.Timedelta(days=900)
    return B
c0 = pd.Timestamp("2025-06-30")
fa, fb = build(T, c0), build(corrupt_future(T, c0), c0)
same = fa.shape == fb.shape and np.allclose(fa.fillna(-999).values.astype(float), fb.fillna(-999).values.astype(float))
check("V2 features", "future-corruption test: features at 2025-06-30 identical when all later data is wrecked", same, f"{fa.shape[0]} patients x {fa.shape[1]} features")

# test 2: no feature is suspiciously close to the target
rho = X.apply(lambda s: s.corr(y, method="spearman")).abs().sort_values(ascending=False)
check("V2 features", "no single feature is a near-copy of the target (|Spearman| < 0.97)", rho.iloc[0] < 0.97, f"strongest = {rho.index[0]} ({rho.iloc[0]:.2f}); averages of many on-drug scores legitimately sit near the target, so this is a red-flag scan, not proof")
check("V2 features", "max feature date <= cutoff", max(T['vis'][T['vis'].visit_date<=END].visit_date.max(), T['sur'].procedure_date.max()) <= END)
print(rho.head(8).round(3).to_string())

features: (21000, 94)  built in 4.2s; uplift per Hurley = {1.0: 2.34, 2.0: 3.48, 3.0: 5.12}
visit_scores       36
visit_behaviour    18
demographics       10
hidden_flare        9
biologic            7
antibiotics         5
surgeries           5
practitioner        3
hurley              1
PASS | V2 features | future-corruption test: features at 2025-06-30 identical when all later data is wrecked | 20255 patients x 94 features
PASS | V2 features | no single feature is a near-copy of the target (|Spearman| < 0.97) | strongest = s_ondrug_mean (0.94); averages of many on-drug scores legitimately sit near the target, so this is a red-flag scan, not proof
PASS | V2 features | max feature date <= cutoff | 
s_ondrug_mean          0.939
s_nonflare_mean_30     0.906
s_mean_30              0.897
s_nonflare_mean_90     0.895
s_min_30               0.893
s_mean_90              0.889
s_nonflare_mean_180    0.887
s_max_30               0.886


# 3 - setting the baseline model
A baseline is needed to compare how the model perform. It is built before any model, and compute every one **out-of-fold** (the group medians and the shrinkage constant are learned on 4/5 of patients and applied to the other 1/5).

| Baseline | Idea |
|---|---|
| B0 population median | no information |
| B1 Hurley median | stage only |
| B2 Hurley x on-biologic median | stage + treatment status (the "simple strong" rule) |
| B3 carry-forward last score | what a clinician might do |
| B4/B5 carry-forward 90d / 180d mean | average of recent scores |
| B6 decayed mean (120d) | recent scores weighted more |
| B7 **shrunk visit level** | patient's decayed *non-flare* score level, pulled toward their B2 group median with weight `w/(w+k)` where `w` is how much (decayed) score evidence exists. Patients with little evidence stay near the group; patients with lots of evidence follow their own scores. `k` is chosen inside each training fold. |

All fall back to B2 when a patient has no score. B7 is the "empirical Bayes" baseline: it is the principled way to use noisy, sparse scores without a model.

In [13]:
# function helpers -- analysis

# take group median
def group_median_oof(keys):
    o = np.full(len(X), np.nan)
    for tr, te in SPL:
        m = y.iloc[tr].groupby([X[k].iloc[tr].values for k in keys]).median()
        idx = pd.MultiIndex.from_arrays([X[k].iloc[te].values for k in keys]) if len(keys) > 1 else pd.Index(X[keys[0]].iloc[te].values)
        o[te] = m.reindex(idx).values
    return pd.Series(o, index=X.index).fillna(y.median())

# shrinking
def shrink(Xs, an, k): 
    w = Xs.s_nonflare_ewm_w_120.fillna(0); return (w * Xs.s_nonflare_ewm_120.fillna(0) + k * an) / (w + k)

In [14]:
# baseline building

SPL = list(KFold(5, shuffle=True, random_state=0).split(np.arange(len(X))))      # the same 5 folds for every baseline and model

B = {}
B["B0 population median"] = pd.Series(np.nan, index=X.index)
for tr, te in SPL: B["B0 population median"].iloc[te] = y.iloc[tr].median()
B["B1 Hurley median"] = group_median_oof(["hurley"])
B["B2 Hurley x biologic median"] = group_median_oof(["hurley", "on_bio"])
anchor = B["B2 Hurley x biologic median"]
for nm, c in [("B3 carry-forward last score", "s_last0"), ("B4 carry-forward 90d mean", "s_mean_90"), ("B5 carry-forward 180d mean", "s_mean_180"), ("B6 decayed mean (120d)", "s_ewm_120")]:
    B[nm] = X[c].fillna(anchor)

o = pd.Series(np.nan, index=X.index); ks = []
for tr, te in SPL:     # k chosen on the training part of the fold only
    k = min([0.5, 1, 2, 3, 4], key=lambda k: mae(y.iloc[tr], shrink(X.iloc[tr], anchor.iloc[tr], k))); ks.append(k)
    o.iloc[te] = shrink(X.iloc[te], anchor.iloc[te], k)
B["B7 shrunk visit level"] = o
EV["eb_ks"] = ks

BASE = pd.DataFrame({k: [mae(y, v)] for k, v in B.items()}, index=["MAE"]).T
print(BASE.round(3).to_string()); print("k chosen per fold:", ks)
E_B = {k: np.abs(y.values - v.values) for k, v in B.items()}
check("V3 baselines", "ladder is sensible: B7 < B2 < B1 < B0", BASE.loc["B7 shrunk visit level","MAE"] < BASE.loc["B2 Hurley x biologic median","MAE"] < BASE.loc["B1 Hurley median","MAE"] < BASE.loc["B0 population median","MAE"])
check("V3 baselines", "raw 'last score' carry-forward is worse than the Hurley x biologic rule (noise + flare bias)", BASE.loc["B3 carry-forward last score","MAE"] > BASE.loc["B2 Hurley x biologic median","MAE"])
EV["base_mae"] = BASE.MAE.to_dict()

                               MAE
B0 population median         4.334
B1 Hurley median             2.667
B2 Hurley x biologic median  2.299
B3 carry-forward last score  2.642
B4 carry-forward 90d mean    2.224
B5 carry-forward 180d mean   2.294
B6 decayed mean (120d)       2.531
B7 shrunk visit level        1.867
k chosen per fold: [1, 1, 1, 1, 1]
PASS | V3 baselines | ladder is sensible: B7 < B2 < B1 < B0 | 
PASS | V3 baselines | raw 'last score' carry-forward is worse than the Hurley x biologic rule (noise + flare bias) | 


# 4 - forecast scorer
**Model:** gradient boosting (`HistGradientBoostingRegressor`) with **absolute-error loss**. The challenge is scored by MAE, and MAE is minimised by the *median*, not the mean; the target is right-skewed (mean 8.3, median 7.0), so squared loss would aim at the wrong place.

**Why no scaling / imputing?** Trees split on thresholds and handle missing values natively. "No score in the last 30 days" is information; filling it with a mean would erase it.

### 4a. What do the visits add?
Each line is a 5-fold out-of-fold MAE on the *same* folds as the baselines. Each patient is predicted by a model that never saw them.

In [15]:
# Function helpers -- analysis

def mk(seed=0): return HGBR(loss="absolute_error", learning_rate=.05, max_iter=400, max_leaf_nodes=24, min_samples_leaf=40, l2_regularization=1., random_state=seed)

def oof(cols, make=mk, splits=SPL, Xd=None, yd=None):
    Xd = X if Xd is None else Xd; yd = y if yd is None else yd; o = np.zeros(len(Xd))
    for tr, te in splits: o[te] = make().fit(Xd.iloc[tr][cols], yd.iloc[tr]).predict(Xd.iloc[te][cols])
    return o

In [16]:

F = lambda *g: [c for c in X.columns if fam[c] in g]
visit_f = F("visit_scores", "visit_behaviour")
core_f = F("hurley", "demographics", "antibiotics", "surgeries", "biologic")   # no visits, no hidden flare, no practitioner
sets = {"A tabular only (no visits)": core_f, "B visit features only": visit_f, "C tabular + visits": core_f + visit_f}
OOF, ERR = {}, {}

for nm, cols in sets.items():
    OOF[nm] = oof(cols); ERR[nm] = np.abs(y.values - OOF[nm]); print(f"{nm:30s} {len(cols):3d} features   MAE {ERR[nm].mean():.4f}")

g = paired_gain(E_B["B7 shrunk visit level"], ERR["C tabular + visits"]); print("gain of C over the best baseline B7:", fmt(g))
g2 = paired_gain(ERR["A tabular only (no visits)"], ERR["C tabular + visits"]); print("gain of visits on top of tabular   :", fmt(g2))
EV["gain_visits"] = g2; EV["gain_vs_B7"] = g
check("V4 forecast", "tabular + visits beats the best baseline (CI above 0)", g["lo"] > 0, fmt(g))

A tabular only (no visits)      28 features   MAE 1.9919
B visit features only           54 features   MAE 1.6977
C tabular + visits              82 features   MAE 1.4401
gain of C over the best baseline B7: +0.4271 (95% CI +0.4104 .. +0.4452)
gain of visits on top of tabular   : +0.5518 (95% CI +0.5322 .. +0.5706)
PASS | V4 forecast | tabular + visits beats the best baseline (CI above 0) | +0.4271 (95% CI +0.4104 .. +0.4452)


### 4b. Ablation: should each *extra* family go in? (hidden flare, practitioner, visit behaviour)
Decision rule used in this notebook: **a family goes into the forecast only if the paired-bootstrap 95% CI of its MAE gain is above 0.** 
For an **artefact-type** family (practitioner habits) we additionally require it to survive *leave-practitioner-out* validation, because a model that leans on site habits breaks when test_2 has different sites.

In [17]:
variants = {"C  base (tabular + visits)": core_f + visit_f,
            "C + hidden flare": core_f + visit_f + F("hidden_flare"),
            "C + practitioner features": core_f + visit_f + F("practitioner"),
            "C + hidden flare + practitioner": core_f + visit_f + F("hidden_flare") + F("practitioner"),
            "C - visit behaviour (no visit counts / flare share / gaps)": core_f + F("visit_scores")}
for nm, cols in variants.items():
    if nm not in OOF: OOF[nm] = oof(cols); ERR[nm] = np.abs(y.values - OOF[nm])
base_e = ERR["C  base (tabular + visits)"]
rows = []
for nm in variants:
    g = paired_gain(base_e, ERR[nm]); rows.append(dict(variant=nm, n_features=len(variants[nm]), MAE=ERR[nm].mean(), gain_vs_base=g["gain"], ci_lo=g["lo"], ci_hi=g["hi"]))
ABL = pd.DataFrame(rows).set_index("variant"); print(ABL.round(4).to_string())
EV["abl"] = ABL

                                                            n_features     MAE  gain_vs_base   ci_lo   ci_hi
variant                                                                                                     
C  base (tabular + visits)                                          82  1.4401        0.0000  0.0000  0.0000
C + hidden flare                                                    91  1.4367        0.0034  0.0001  0.0068
C + practitioner features                                           85  1.4404       -0.0004 -0.0025  0.0017
C + hidden flare + practitioner                                     94  1.4390        0.0010 -0.0022  0.0042
C - visit behaviour (no visit counts / flare share / gaps)          64  1.4480       -0.0079 -0.0117 -0.0041


### 4c. Hidden-flare component: does it carry real information, and for whom?
Three questions before trusting it:
1. **Is the hidden count mostly a *documentation* signal?** If low-documentation practitioners simply have more "unseen" courses, the feature could be measuring paperwork. We split patients by their practitioner's scoring rate.
2. **Do low-documentation sites have systematically higher residuals** (the base model *under*-predicts them) -- which is what real unseen flares would cause?
3. **Does a simple additive rule work?** `shrunk visit level + expected hidden burden`. If hidden flares were simply additive and well measured, this rule would improve on B7.

In [18]:
rate = X.p_score_rate; terc = pd.qcut(rate, 3, labels=["low documentation", "mid", "high documentation"])
e_b, e_h = ERR["C  base (tabular + visits)"], ERR["C + hidden flare"]
rows = []
for t_ in terc.cat.categories:
    m = (terc == t_).values; g = paired_gain(e_b[m], e_h[m], n=500)
    rows.append(dict(tercile=t_, n=int(m.sum()), hidden_courses_per_patient=float(X.hid_abx_365[m].mean()), cover=float(X.flare_cover_abx_365[m].mean()), base_MAE=e_b[m].mean(), gain=g["gain"], lo=g["lo"], hi=g["hi"]))
HF = pd.DataFrame(rows).set_index("tercile"); print(HF.round(4).to_string()); EV["hf_terc"] = HF

# residual of the model WITHOUT hidden features vs the practitioner's documentation
res_base = pd.Series(y.values - OOF["C  base (tabular + visits)"], index=X.index)
pr = pd.DataFrame({"res": res_base, "rate": rate, "pid": X.prac_num}).groupby("pid").agg(res=("res", "mean"), rate=("rate", "first"), n=("res", "size"))
pr = pr[pr.n >= 10]; r_pr = stats.spearmanr(pr.rate, pr.res)
EV["prac_rate_res_rho"] = float(r_pr.statistic); EV["prac_rate_res_p"] = float(r_pr.pvalue)
print(f"\nacross {len(pr)} practitioners (>=10 patients): Spearman(scoring rate, mean residual of base model) = {r_pr.statistic:+.3f} (p={r_pr.pvalue:.3g})")

# additive structural rule
w = X.s_nonflare_ewm_w_120.fillna(0); k = float(np.median(EV["eb_ks"]))
lvl = (w * X.s_nonflare_ewm_120.fillna(0) + k * B["B2 Hurley x biologic median"]) / (w + k)
EV["additive_mae"] = {a: mae(y, lvl + a * X.hid_burden_365) for a in (0, 0.5, 1, 2)}
print("additive rule MAE (0 = plain B7 level, a = weight on expected hidden burden):", {a: round(v, 4) for a, v in EV["additive_mae"].items()})

                       n  hidden_courses_per_patient   cover  base_MAE    gain      lo      hi
tercile                                                                                       
low documentation   7003                      0.7314  0.5163    1.4898  0.0056 -0.0009  0.0108
mid                 6998                      0.5620  0.6325    1.4548  0.0025 -0.0029  0.0082
high documentation  6998                      0.4110  0.7278    1.3755  0.0020 -0.0032  0.0073

across 1058 practitioners (>=10 patients): Spearman(scoring rate, mean residual of base model) = -0.043 (p=0.159)
additive rule MAE (0 = plain B7 level, a = weight on expected hidden burden): {0: 1.8672, 0.5: 1.8739, 1: 1.8824, 2: 1.9041}


### 4d. Decisions from the ablation (written to `CFG`)
The decision rule is turned into code so the choice is reproducible.

In [19]:
# function helpers -- analysis

def keep(variant):  # CI lower bound above zero
    return bool(ABL.loc[variant, "ci_lo"] > 0)

def feature_cols(cfg=None):
    cfg = cfg or CFG
    cols = core_f + visit_f
    if cfg["use_hidden_flare"]: cols = cols + F("hidden_flare")
    if cfg["use_practitioner"]: cols = cols + F("practitioner")
    return cols

In [20]:
CFG["use_hidden_flare"] = keep("C + hidden flare")
prac_ok = keep("C + practitioner features")  # artefact family: must ALSO survive leave-practitioner-out (checked in step 6)
CFG["use_practitioner"] = False   # default off; step 6 re-opens the question
print("use_hidden_flare =", CFG["use_hidden_flare"], "| practitioner features pass the first gate:", prac_ok)

FEATS = feature_cols(); print(len(FEATS), "features in the forecast core")

use_hidden_flare = True | practitioner features pass the first gate: False
91 features in the forecast core


# 5 - Validation layer for the forecast 
Random 5-fold CV answers "does the model work on patients like these?". 
It does **not** answer: *does it find real signal, does it depend on site habits, does it survive time, does it survive a different population?* etc. Each question gets its own test.

| Test | Question | Pass condition |
|---|---|---|
| 6a shuffled-target null | Is the pipeline able to "learn" from noise (a leak or overfit)? | model on shuffled targets is no better than the median |
| 6b leave-practitioner-out | Does the model lean on site habits? | MAE barely changes when whole practitioners are held out |
| 6c time-forward backtest | Does a model trained on the past predict a later period? | beats baselines at the later cutoff, on unseen patients |
| 6d shift stress tests | What happens when the population differs (the test_2 situation)? | extra error stays small, and we know which features drift |
| 6e error slices | Where is the model weak or biased? | no slice is badly biased |

In [ ]:
# 5a: shuffled-target null
rng = np.random.default_rng(0); y_sh = pd.Series(rng.permutation(y.values), index=y.index)
o_null = oof(FEATS, yd=y_sh); m_null, m_med = mae(y_sh, o_null), mae(y_sh, y_sh.median())
check("V5 core", "shuffled-target null: model cannot beat the median on shuffled labels", m_null > 0.97 * m_med, f"null MAE {m_null:.3f} vs median {m_med:.3f}")

PASS | V5 core | shuffled-target null: model cannot beat the median on shuffled labels | null MAE 4.338 vs median 4.334


In [ ]:
# 5b: leave-practitioner-out
grp = X.prac_num.fillna(-1).values
SPL_G = list(GroupKFold(5).split(X, groups=grp))
res = {}
for nm, cols in {"forecast core": FEATS, "core + practitioner features": FEATS + [c for c in F("practitioner") if c not in FEATS]}.items():
    r_ = oof(cols); g_ = oof(cols, splits=SPL_G)
    res[nm] = dict(random_cv_MAE=mae(y, r_), practitioner_held_out_MAE=mae(y, g_), e_rand=np.abs(y.values - r_), e_grp=np.abs(y.values - g_))
tab = pd.DataFrame({k: {a: b for a, b in v.items() if a.endswith("MAE")} for k, v in res.items()}).T
tab["penalty"] = tab.practitioner_held_out_MAE - tab.random_cv_MAE; print(tab.round(4).to_string())
gp = paired_gain(res["forecast core"]["e_grp"], res["core + practitioner features"]["e_grp"])
EV["prac_gain_heldout"] = gp; EV["prac_penalty"] = float(tab.loc["forecast core", "penalty"])
print("practitioner features, gain when practitioners are held out:", fmt(gp))
check("V5 core", "leave-practitioner-out penalty is small (< 0.03 MAE)", tab.loc["forecast core", "penalty"] < 0.03, f"{tab.loc['forecast core','penalty']:+.4f}")
CFG["use_practitioner"] = bool(prac_ok and gp["lo"] > 0)       # artefact family needs BOTH gates
print("final: use_practitioner =", CFG["use_practitioner"], "| use_hidden_flare =", CFG["use_hidden_flare"])
FEATS = feature_cols(); OOF["FINAL single"] = oof(FEATS); ERR["FINAL single"] = np.abs(y.values - OOF["FINAL single"])

                              random_cv_MAE  practitioner_held_out_MAE  penalty
forecast core                        1.4367                     1.4373   0.0005
core + practitioner features         1.4390                     1.4407   0.0016
practitioner features, gain when practitioners are held out: -0.0034 (95% CI -0.0058 .. -0.0009)
PASS | V5 core | leave-practitioner-out penalty is small (< 0.03 MAE) | +0.0005
final: use_practitioner = False | use_hidden_flare = True


5c. Time-forward backtest (the real target exists for one date only, so it needs pseudo-labels for past dates)
For an old cutoff `c`, build features as of `c`, and use the **mean of the scored visits in the next 6 months** as a noisy label (dropping patients who start/stop a biologic inside the window, like the challenge). Train on *earlier* cutoffs and **half the patients**, test on a **later cutoff and the other half of the patients**: separated in time *and* in patients, so neither temporal nor patient leakage can help.

**Limits to remember:** the label only contains *observed* scores, so it is noisy and flare-biased and is **blind to hidden flares**. Use it for *relative* comparisons (model vs baselines, ranking of variants), never for absolute MAE, and **not to judge the hidden-flare block** (the label cannot contain what was never scored).

In [ ]:
# 5c. time forward backtest
def labels(c, months=6):
    c = pd.Timestamp(c); e = c + pd.DateOffset(months=months)
    w = T["vis"][(T["vis"].visit_date > c) & (T["vis"].visit_date <= e) & T["vis"].ihs4_score.notna()]
    yy = w.groupby("patient_id").ihs4_score.agg(["mean", "size"])
    b = T["trt"][T["trt"].treatment_class == "Biologic"]
    chg = set(b[(b.start_date > c) & (b.start_date <= e)].patient_id) | set(b[(b.end_date > c) & (b.end_date <= e)].patient_id)
    return yy[~yy.index.isin(chg)]
BT = {}
def bt_rows(c):
    c = str(pd.Timestamp(c).date())
    if c not in BT: BT[c] = build(T, c).join(labels(c), how="inner").assign(cut=c)
    return BT[c]

train_cuts = ["2022-06-30", "2022-12-31", "2023-06-30", "2023-12-31", "2024-06-30"]; test_cut = "2024-12-31"
t0 = time.time(); trn = pd.concat([bt_rows(c) for c in train_cuts]); tst = bt_rows(test_cut); print(f"built backtest features in {time.time()-t0:.0f}s")
rs = np.random.default_rng(1).permutation(T["dem"].patient_id.values); A_ = set(rs[:len(rs) // 2])      # half of ALL patients for training, the other half for testing

trn = trn[trn.index.isin(A_)]; tst = tst[~tst.index.isin(A_)]
check("V6 time-forward", "train and test patients are disjoint", len(set(trn.index) & set(tst.index)) == 0, f"train rows {len(trn)}, test rows {len(tst)}")
check("V6 time-forward", "last training label window ends on/before the test cutoff", pd.Timestamp(train_cuts[-1]) + pd.DateOffset(months=6) <= pd.Timestamp(test_cut))
yt = tst["mean"]

anc = trn.groupby(["hurley", "on_bio"])["mean"].median(); fb = pd.Series([anc.get((h, b), trn["mean"].median()) for h, b in zip(tst.hurley, tst.on_bio)], index=tst.index)
kk = float(np.median(EV["eb_ks"])); w_ = tst.s_nonflare_ewm_w_120.fillna(0)

out = {"Hurley x biologic median": mae(yt, fb), "carry-forward last score": mae(yt, tst.s_last0.fillna(fb)), "decayed mean (120d)": mae(yt, tst.s_ewm_120.fillna(fb)),
       "shrunk visit level (B7)": mae(yt, (w_ * tst.s_nonflare_ewm_120.fillna(0) + kk * fb) / (w_ + kk))}

for nm, cols in {"GBM tabular only": core_f, "GBM tabular + visits": core_f + visit_f}.items():
    out[nm] = mae(yt, mk().fit(trn[cols], trn["mean"]).predict(tst[cols]))
BTR = pd.Series(out); print(BTR.round(3).to_string()); EV["bt"] = BTR
check("V6 time-forward", "forecast core beats every baseline on a later date and unseen patients", BTR["GBM tabular + visits"] < BTR.drop(["GBM tabular + visits", "GBM tabular only"]).min(), f"{BTR['GBM tabular + visits']:.3f} vs best baseline {BTR.drop(['GBM tabular + visits','GBM tabular only']).min():.3f}")

built backtest features in 21s
PASS | V6 time-forward | train and test patients are disjoint | train rows 24602, test rows 5812
PASS | V6 time-forward | last training label window ends on/before the test cutoff | 
Hurley x biologic median    3.081
carry-forward last score    2.769
decayed mean (120d)         2.574
shrunk visit level (B7)     2.445
GBM tabular only            2.790
GBM tabular + visits        2.115
PASS | V6 time-forward | forecast core beats every baseline on a later date and unseen patients | 2.115 vs best baseline 2.445


5d. Shift stress tests and the test-set shift tool
build the **tool** that compares any new set to train, and **simulate** shifts inside train: train the model on one kind of patient, test on a different kind, and measure the extra error versus random CV *on those same patients*.

AUC 0.5 = indistinguishable; AUC near 1 = very different. The features that make it separable show *where* the shift is. A high AUC does not mean the model will fail; the stress test shows whether it does.

In [ ]:
# function helpers -- analysis shift stress test

def drift_table(Xa, Xb, cols, top=10):
    # Kolmogorov-Smirnov distance per feature (0 = same distribution) and the missing-rate gap
    rows = []
    for c in cols:
        a, b = Xa[c].dropna(), Xb[c].dropna()
        if len(a) > 20 and len(b) > 20: rows.append((c, stats.ks_2samp(a, b).statistic, Xb[c].isna().mean() - Xa[c].isna().mean()))
    return pd.DataFrame(rows, columns=["feature", "KS", "missing_rate_gap"]).set_index("feature").sort_values("KS", ascending=False).head(top)

def adversarial_validation(Xa, Xb, cols, seed=0):
    Z = pd.concat([Xa[cols], Xb[cols]]); lab = np.r_[np.zeros(len(Xa)), np.ones(len(Xb))]
    o = np.zeros(len(Z))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Z, lab):
        o[te] = HGBC(max_iter=150, learning_rate=.08, max_leaf_nodes=15, random_state=0).fit(Z.iloc[tr], lab[tr]).predict_proba(Z.iloc[te])[:, 1]
    return roc_auc_score(lab, o)

def shift_report(X_train, X_new, cols, name="new set"):
    auc = adversarial_validation(X_train, X_new, cols)
    print(f"[{name}] adversarial AUC train-vs-new = {auc:.3f}  (0.5 = same population)")
    return auc, drift_table(X_train, X_new, cols)

In [ ]:
# 5d. shift stress test

dem_ = T["dem"].set_index("patient_id").reindex(X.index)
dx_year = dem_.diagnosis_date.dt.year; eth = dem_.ethnicity

scen = {"unseen practitioners (5-fold group CV)": None,
        "older diagnosis cohort -> newer (dx <=2020 vs >=2021)": (dx_year <= 2020, dx_year >= 2021),
        "younger -> older (age <= median vs above)": (X.age <= X.age.median(), X.age > X.age.median()),
        "White/unknown -> other ethnicities": (eth.isin(["White"]) | eth.isna(), ~(eth.isin(["White"]) | eth.isna())),
        "severity extrapolation: train on Hurley 1+2 -> test on Hurley 3": (X.hurley <= 2, X.hurley == 3)}

rows = []; e_rand = ERR["FINAL single"]; 
for nm, s in scen.items():
    if s is None:
        o = oof(FEATS, splits=SPL_G); te_mask = np.ones(len(X), bool); e_shift = np.abs(y.values - o); auc = np.nan
    else:
        tr_m, te_m = s; tr_i, te_i = np.where(tr_m.values)[0], np.where(te_m.values)[0]
        p = mk().fit(X.iloc[tr_i][FEATS], y.iloc[tr_i]).predict(X.iloc[te_i][FEATS]); te_mask = np.zeros(len(X), bool); te_mask[te_i] = True
        e_shift = np.full(len(X), np.nan); e_shift[te_i] = np.abs(y.values[te_i] - p)
        auc = adversarial_validation(X.iloc[tr_i], X.iloc[te_i], FEATS)
    b7 = E_B["B7 shrunk visit level"]
    rows.append(dict(scenario=nm, n_test=int(te_mask.sum()), model_MAE_random_CV=e_rand[te_mask].mean(), model_MAE_shifted=np.nanmean(e_shift[te_mask]),
                     B7_MAE=b7[te_mask].mean(), extra_error=np.nanmean(e_shift[te_mask]) - e_rand[te_mask].mean(), adv_AUC=auc))

SH = pd.DataFrame(rows).set_index("scenario"); print(SH.round(3).to_string()); EV["shift"] = SH
real = SH.index[~SH.index.str.startswith("severity")]

check("V7 shift", "under every population shift (practitioners, cohort era, age, ethnicity) the model still beats the shrunk-visit baseline B7", bool((SH.loc[real].model_MAE_shifted < SH.loc[real].B7_MAE).all()))

check("V7 shift", "extra error under those population shifts stays below 0.3 MAE", SH.loc[real].extra_error.max() < 0.3, f"max {SH.loc[real].extra_error.max():+.3f}")
sev = SH.loc[SH.index.str.startswith("severity")].iloc[0]

check("V7 shift", "a tree model can extrapolate to a severity level it never saw", bool(sev.model_MAE_shifted < sev.B7_MAE), f"train on Hurley 1+2, test on Hurley 3: model {sev.model_MAE_shifted:.2f} vs B7 {sev.B7_MAE:.2f}. Only a problem if test_2 contains a severity range absent from train", warn=True)

                                                                 n_test  model_MAE_random_CV  model_MAE_shifted  B7_MAE  extra_error  adv_AUC
scenario                                                                                                                                     
unseen practitioners (5-fold group CV)                            21000                1.437              1.437   1.867        0.001      NaN
older diagnosis cohort -> newer (dx <=2020 vs >=2021)             12624                1.365              1.547   1.830        0.181      1.0
younger -> older (age <= median vs above)                          9532                1.429              1.453   1.824        0.024      1.0
White/unknown -> other ethnicities                                10056                1.461              1.479   1.893        0.017      1.0
severity extrapolation: train on Hurley 1+2 -> test on Hurley 3    5155                1.701              3.379   2.331        1.678      1.0
PASS |

5e. Where is the model weak or biased? (error slices)
`bias` = mean(prediction - truth). A median-loss model on a right-skewed target will sit a little low overall; what matters is that no slice is far worse than the rest. The "no scored visit" and "low documentation" slices are the ones most exposed to the recording artefacts.

In [ ]:
# model bias and weakness analysis
e = pd.Series(ERR["FINAL single"], index=X.index); sgn = pd.Series(OOF["FINAL single"] - y.values, index=X.index)
nsc = (X.s_mean_365.isna()).map({True: "no scored visit in last 365d", False: "has scored visit"})
slices = {"Hurley": X.hurley, "on biologic": X.on_bio, "documentation tercile": terc, "scored visits in last year": nsc,
          "prevalent cohort (dx placeholder)": X.prevalent, "days since last visit": pd.cut(X.v_days_since, [-1, 30, 90, 180, 365, 5000])}

rows = []
for k, s in slices.items():
    for lv, idx in e.groupby(s).groups.items(): rows.append(dict(slice=k, level=str(lv), n=len(idx), MAE=e[idx].mean(), bias=sgn[idx].mean(), B7_MAE=E_B["B7 shrunk visit level"][X.index.get_indexer(idx)].mean()))

SL = pd.DataFrame(rows).set_index(["slice", "level"]); print(SL.round(3).to_string()); EV["slices"] = SL
check("V8 slices", "model beats B7 in every slice with >= 300 patients", bool((SL[SL.n >= 300].MAE < SL[SL.n >= 300].B7_MAE).all()))
check("V8 slices", "no slice has |bias| > 1.0 IHS4 point", SL.bias.abs().max() < 1.0, f"max |bias| {SL.bias.abs().max():.2f}")

                                                                    n    MAE   bias  B7_MAE
slice                             level                                                    
Hurley                            1.0                            6280  1.225 -0.176   1.432
                                  2.0                            9565  1.433 -0.266   1.902
                                  3.0                            5155  1.701 -0.340   2.331
on biologic                       0.0                           12452  1.616 -0.390   1.908
                                  1.0                            8548  1.176 -0.064   1.807
documentation tercile             low documentation              7003  1.484 -0.263   1.902
                                  mid                            6998  1.452 -0.289   1.872
                                  high documentation             6998  1.373 -0.220   1.828
scored visits in last year        has scored visit              16932  1.402 -0.

5f. Final forecast blend 
Three variants are averaged: absolute-loss, a slower/regularised absolute-loss model, and a squared-loss model on `log1p(target)` (back-transformed). Averaging models that make different small mistakes lowers variance. It is evaluated out-of-fold with the same folds

In [ ]:
# function helpers - analysis blender

def make_models():
    return [mk(0),
            HGBR(loss="absolute_error", learning_rate=.03, max_iter=900, max_leaf_nodes=16, min_samples_leaf=60, l2_regularization=3., random_state=0),
            HGBR(loss="squared_error", learning_rate=.04, max_iter=600, max_leaf_nodes=16, min_samples_leaf=50, l2_regularization=3., random_state=0)]

def fit_blend(Xt, yt, cols):
    ms = make_models(); ms[0].fit(Xt[cols], yt); ms[1].fit(Xt[cols], yt); ms[2].fit(Xt[cols], np.log1p(yt)); return ms

def predict_blend(ms, Xn, cols):
    p = [ms[0].predict(Xn[cols]), ms[1].predict(Xn[cols]), np.expm1(ms[2].predict(Xn[cols]))]; return np.clip(np.mean(p, axis=0), 0, None)

In [ ]:
# final forecast blend

o_bl = np.zeros(len(X))
for tr, te in SPL: o_bl[te] = predict_blend(fit_blend(X.iloc[tr], y.iloc[tr], FEATS), X.iloc[te], FEATS)
OOF["M0 blend"] = o_bl; ERR["M0 blend"] = np.abs(y.values - o_bl)
bio = (X.on_bio == 1).values

print(f"M0 blend MAE (all 21,000)            : {ERR['M0 blend'].mean():.4f}   (single model {ERR['FINAL single'].mean():.4f}, best baseline B7 {E_B['B7 shrunk visit level'].mean():.4f})")

print(f"Exercise 2a: on-biologic patients     : {ERR['M0 blend'][bio].mean():.4f}   (n={bio.sum()}; B7 {E_B['B7 shrunk visit level'][bio].mean():.4f})")

print(f"             not on biologic          : {ERR['M0 blend'][~bio].mean():.4f}")
EV["m0_mae"] = float(ERR["M0 blend"].mean()); EV["m0_mae_bio"] = float(ERR["M0 blend"][bio].mean()); EV["m0_vs_B7"] = paired_gain(E_B["B7 shrunk visit level"], ERR["M0 blend"])

print("gain over B7:", fmt(EV["m0_vs_B7"]))
check("V9 blend", "blend is not worse than the best single model (within CI)", paired_gain(ERR["FINAL single"], ERR["M0 blend"])["hi"] > 0)

M0 blend MAE (all 21,000)            : 1.4306   (single model 1.4367, best baseline B7 1.8672)
Exercise 2a: on-biologic patients     : 1.1677   (n=8548; B7 1.8072)
             not on biologic          : 1.6110
gain over B7: +0.4366 (95% CI +0.4193 .. +0.4542)
PASS | V9 blend | blend is not worse than the best single model (within CI) | 


# 6 - classification for explanation and problem exercise
This section:
1. builds the **response types** (deep / partial / minimal responders) and checks they are real, not noise;
2. runs the **ablation gate**: add the response-type features to the forecast and see whether MAE drops;
3. builds the **super-responder (2b) analysis** with *pre-treatment-only* information, which answers "can we spot them before or early in treatment?" (the earlier AUC of 0.97 was circular: the label came from the target and the model saw on-treatment scores).

6a. Response types (first-line biologic patients)
For each patient on a first-line biologic: **pre-drug level** = mean score in the 365 days before the line started; **on-drug level** = mean score from day 90 on (the drug has had time to work). Both are noisy, based on a few visits, so each is **shrunk toward its Hurley-stage mean** (empirical Bayes) in proportion to how few visits it rests on. This avoids *regression to the mean*: a patient with one very high pre-drug score would otherwise look like a great "responder" just because extreme values fall back.
`ratio = on-drug / pre-drug`. A 3-component Gaussian mixture on `log(ratio)` gives the types. The target is **never used** to define the types.

In [29]:
# functions helper
# response types
def cur_lines(Tc):
    b = Tc["trt"][(Tc["trt"].treatment_class == "Biologic") & Tc["trt"].end_date.isna()]
    return b.sort_values("start_date").groupby("patient_id").tail(1).set_index("patient_id")
def eb_shrink(m, n, v, group):
    """Empirical-Bayes shrink patient means toward their group mean (noise-aware)."""
    out = pd.Series(np.nan, index=m.index); 
    for g in group.dropna().unique():
        i = group == g; mm, nn, vv = m[i], n[i], v[i]
        mu = mm.mean(); s2w = np.nanmean(vv[nn >= 2]); s2b = max(mm.var() - np.nanmean(s2w / nn), 1e-3)
        k = s2w / s2b; out[i] = (nn * mm + k * mu) / (nn + k)
    return out
def response_table(Tc, hurley):
    cur = cur_lines(Tc); cur = cur[cur.treatment_line == 1]
    v = Tc["vis"][Tc["vis"].ihs4_score.notna()].join(cur.start_date.rename("t0"), on="patient_id", how="inner")
    d = (v.visit_date - v.t0).dt.days
    parts = {}
    for nm, x in [("pre", v[(d < 0) & (d >= -365)]), ("on", v[d >= 90])]:
        g = x.groupby("patient_id").ihs4_score; parts[nm] = pd.DataFrame({f"{nm}_n": g.size(), f"{nm}_m": g.mean(), f"{nm}_v": g.var()})
    r = pd.concat(parts.values(), axis=1, join="inner")
    h = hurley.reindex(r.index)
    r["pre_s"] = eb_shrink(r.pre_m, r.pre_n, r.pre_v, h); r["on_s"] = eb_shrink(r.on_m, r.on_n, r.on_v, h)
    r["ratio"] = r.on_s / r.pre_s; r["lr"] = np.log(r.ratio)
    return r, cur

# super responder analyser
def pretx_feats(Tc, f, early=True):
    """Features frozen at the start t0 of each patient's current biologic line (what a clinician knows BEFORE treating),
    plus optional early-response features from the first 6 months on drug."""
    cur = cur_lines(Tc); t0 = cur.start_date
    out = pd.DataFrame(index=cur.index)
    for c_ in ["hurley","sex","bmi","smoking_status","ethnicity","family_history_hs","prevalent","age_dx","delay"]: out[c_] = f[c_].reindex(out.index)
    out["line"] = cur.treatment_line; out["drug"] = cur.drug_name.astype("category").cat.codes.astype(float)
    out["start_year"] = t0.dt.year; out["age_start"] = (t0.dt.year - (Tc["dem"].set_index("patient_id").birth_year.reindex(out.index)))
    dx = Tc["dem"].set_index("patient_id").diagnosis_date.reindex(out.index)
    out["dx_to_start_days"] = (t0 - dx).dt.days.where(out.prevalent == 0)
    for nm, tab, col, flt in [("abx", Tc["trt"], "start_date", Tc["trt"].treatment_class == "Antibiotic"), ("sur", Tc["sur"], "procedure_date", slice(None))]:
        x = tab[flt] if not isinstance(flt, slice) else tab
        x = x.join(t0.rename("t0"), on="patient_id", how="inner"); d = (x.t0 - x[col]).dt.days
        for w in (180, 365):
            out[f"pre_{nm}_{w}"] = x[(d > 0) & (d <= w)].groupby("patient_id").size()
    v = Tc["vis"].join(t0.rename("t0"), on="patient_id", how="inner"); d = (v.visit_date - v.t0).dt.days
    pre = v[(d < 0) & (d >= -365)]; s = pre[pre.ihs4_score.notna()].groupby("patient_id").ihs4_score
    out["pre_s_mean"] = s.mean(); out["pre_s_max"] = s.max(); out["pre_s_n"] = s.size(); out["pre_v_n"] = pre.groupby("patient_id").size()
    out["pre_flare_share"] = pre.assign(fl=pre.visit_type == "flare_reactive").groupby("patient_id").fl.mean()
    zc = [k for k in out if k.startswith(("pre_abx","pre_sur","pre_v_n","pre_s_n"))]; out[zc] = out[zc].fillna(0)
    if early:
        ok = (t0 + pd.Timedelta(days=180)) <= END
        e = v[(d >= 90) & (d < 180)]; es = e[e.ihs4_score.notna()].groupby("patient_id").ihs4_score.mean()
        out["early_s_m3_6"] = es.where(ok.reindex(es.index)); 
        for nm, tab, col, flt in [("abx", Tc["trt"], "start_date", Tc["trt"].treatment_class == "Antibiotic"), ("sur", Tc["sur"], "procedure_date", slice(None))]:
            x = tab[flt] if not isinstance(flt, slice) else tab
            x = x.join(t0.rename("t0"), on="patient_id", how="inner"); dd = (x[col] - x.t0).dt.days
            out[f"early_{nm}_6m"] = x[(dd >= 0) & (dd < 180)].groupby("patient_id").size().reindex(out.index).fillna(0).where(ok)
    return out

# AUC analysis
def auc_cv(cols, reps=3):
    a, oofp = [], np.zeros(len(P))
    for s in range(reps):
        o = np.zeros(len(P))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=s).split(P, P.sr):
            o[te] = HGBC(max_iter=200, learning_rate=.05, max_leaf_nodes=8, min_samples_leaf=40, random_state=0).fit(P.iloc[tr][cols], P.sr.iloc[tr]).predict_proba(P.iloc[te][cols])[:, 1]
        a.append(roc_auc_score(P.sr, o)); oofp += o / reps
    return np.mean(a), np.std(a), oofp

In [30]:
# response type analysis
Tc = as_of(T, END); hurley_all = f_all.hurley
RT, CUR = response_table(Tc, hurley_all)
gm = GaussianMixture(3, random_state=0, n_init=5).fit(RT[["lr"]])
order = np.argsort(gm.means_.ravel())  # most negative log-ratio = deepest response
names = np.array(["deep", "partial", "minimal"]); rank = np.empty(3, int); rank[order] = np.arange(3)
post = gm.predict_proba(RT[["lr"]])[:, order]                          # columns: deep, partial, minimal

RT["type"] = names[post.argmax(1)]; RT["p_deep"], RT["p_partial"] = post[:, 0], post[:, 1]
print("patients with a first-line biologic and enough scores:", len(RT), "of", len(CUR), "first-line patients")
print(RT.type.value_counts(normalize=True).round(3).to_string())
print("typical on/pre ratio by type:", RT.groupby("type").ratio.median().round(2).to_dict())

# validation 1: types predict the (unused) target
chk = RT.join(y, how="inner").assign(h=hurley_all)
tt = chk.groupby(["h", "type"]).avg_ihs4_next6.agg(["mean", "size"]).round(2); print("\nmean target by Hurley x type (target NOT used to define types):\n", tt.to_string())

# validation 2: split-half stability (odd visits vs even visits) - are types a stable patient property or noise?
v_ = Tc["vis"][Tc["vis"].ihs4_score.notna()].join(CUR[CUR.treatment_line == 1].start_date.rename("t0"), on="patient_id", how="inner")
v_["d"] = (v_.visit_date - v_.t0).dt.days; v_["k"] = v_.groupby("patient_id").cumcount() % 2
lr_h = {k: np.log(v_[(v_.k == k) & (v_.d >= 90)].groupby("patient_id").ihs4_score.mean() / v_[(v_.k == k) & (v_.d < 0) & (v_.d >= -365)].groupby("patient_id").ihs4_score.mean()) for k in (0, 1)}
sh = pd.concat(lr_h, axis=1).replace([np.inf, -np.inf], np.nan).dropna().corr().iloc[0, 1]; EV["resp_split_half"] = float(sh); EV["resp_shares"] = RT.type.value_counts(normalize=True).to_dict()
EV["resp_target_gap"] = float(tt.loc[(2.0, "minimal"), "mean"] - tt.loc[(2.0, "deep"), "mean"])
check("V10 classification", "response ratio is stable across independent halves of the visits (r > 0.5)", sh > 0.5, f"split-half r = {sh:.2f}")
check("V10 classification", "types order the unused target (deep < partial < minimal in Hurley 2)", tt.loc[(2.0, "deep"), "mean"] < tt.loc[(2.0, "partial"), "mean"] < tt.loc[(2.0, "minimal"), "mean"])

patients with a first-line biologic and enough scores: 4741 of 6762 first-line patients
type
minimal    0.622
partial    0.232
deep       0.146
typical on/pre ratio by type: {'deep': 0.2, 'minimal': 0.77, 'partial': 0.4}

mean target by Hurley x type (target NOT used to define types):
               mean  size
h   type                
2.0 deep      1.89   633
    minimal   7.41  1698
    partial   4.01   838
3.0 deep      3.96    60
    minimal  14.04  1249
    partial   8.21   263
PASS | V10 classification | response ratio is stable across independent halves of the visits (r > 0.5) | split-half r = 0.85
PASS | V10 classification | types order the unused target (deep < partial < minimal in Hurley 2) | 


6b. The ablation gate: do response types lower the forecast MAE?
Add `resp_ratio`, `resp_p_deep`, `resp_p_partial` (missing for everyone who is not a first-line biologic patient) to the forecast features and compare out-of-fold MAE with the same folds. Expectation from the medication-history result: **no gain**, because the ratio is a function of on-drug and pre-drug visit scores, and the model already sees on-drug scores (`s_ondrug_mean`, decayed means).

In [31]:
# forecast MAE for each response types
Xr = X.copy(); Xr["resp_ratio"] = RT.ratio.reindex(X.index); Xr["resp_p_deep"] = RT.p_deep.reindex(X.index); Xr["resp_p_partial"] = RT.p_partial.reindex(X.index)
RESP_F = ["resp_ratio", "resp_p_deep", "resp_p_partial"]

o_r = oof(FEATS + RESP_F, Xd=Xr); e_r = np.abs(y.values - o_r)
sub = Xr.resp_ratio.notna().values  # first-line biologic patients with a response ratio
g_all = paired_gain(ERR["FINAL single"], e_r); g_sub = paired_gain(ERR["FINAL single"][sub], e_r[sub])

print(f"MAE without response features {ERR['FINAL single'].mean():.4f} | with {e_r.mean():.4f}")

print("gain, all patients      :", fmt(g_all)); print(f"gain, first-line biologic ({sub.sum()}):", fmt(g_sub))
EV["resp_gate_all"], EV["resp_gate_sub"] = g_all, g_sub
CFG["use_response_types"] = bool(g_all["lo"] > 0)

print("-> use_response_types =", CFG["use_response_types"], "(response types stay out of the forecast)" if not CFG["use_response_types"] else "")
check("V10 classification", "ablation gate applied: response types enter the forecast only with CI lower bound > 0", True, f"decision: {CFG['use_response_types']}")

MAE without response features 1.4367 | with 1.4390
gain, all patients      : -0.0023 (95% CI -0.0045 .. +0.0001)
gain, first-line biologic (4741): -0.0041 (95% CI -0.0085 .. +0.0004)
-> use_response_types = False (response types stay out of the forecast)
PASS | V10 classification | ablation gate applied: response types enter the forecast only with CI lower bound > 0 | decision: False


7c. Super-responders: what can be known **before** or **early in** treatment?
**Label (a proxy, not the truth).** The challenge says roughly 1 in 8 biologic patients is a super-responder (IHS4 more than 75% below what it would have been untreated). The untreated counterfactual is not observed, so we use: *target < 25% of the mean target of non-biologic patients with the same Hurley stage*. It is a **group** comparison, not each patient's own counterfactual, so it is a proxy; it matches the stated rate (about 12%).

**Three information levels**, each cross-validated (stratified 5-fold, repeated 3 times):
1. **Before treatment:** demography + Hurley + everything known on the day the current biologic started (history in the 365 days before).
2. **Early in treatment:** add the first 6 months on drug (months 3-6 scores, antibiotics, surgeries). Only available for those who started >= 6 months before the cutoff.
3. **Circular reference:** the forecast itself used as a detector (its AUC is high only because the label *is* the target).

In [32]:
# super responder analysis

P = pretx_feats(Tc, f_all).join(y, how="inner")
nb_mean = y[(X.on_bio == 0)].groupby(X.hurley[(X.on_bio == 0)]).mean()                # non-biologic mean target per Hurley stage
P["sr"] = (P.avg_ihs4_next6 < 0.25 * P.hurley.map(nb_mean)).astype(int)

print(f"{len(P)} biologic patients with a target | proxy super-responders: {P.sr.mean():.1%} (challenge brief: ~1 in 8 = 12.5%)  | by Hurley: {P.groupby('hurley').sr.mean().round(3).to_dict()}")


base_c = ["hurley", "sex", "bmi", "smoking_status", "ethnicity", "family_history_hs", "prevalent", "age_dx", "delay"]
pre_c = [c for c in P if not c.startswith("early") and c not in ("avg_ihs4_next6", "sr")]; early_c = [c for c in P if c.startswith("early")]

SR = {}
for nm, cols in {"1a demography + Hurley": base_c, "1b + pre-treatment history (at line start)": pre_c, "2  + first 6 months on drug": pre_c + early_c}.items():
    m_, s_, o_ = auc_cv(cols); SR[nm] = (m_, s_); print(f"{nm:45s} AUC {m_:.3f} +- {s_:.3f}"); 
    if nm.startswith("1b"): P["p_pre"] = o_
m0 = pd.Series(OOF["M0 blend"], index=X.index).reindex(P.index); SR["3  forecast used as detector (circular)"] = (roc_auc_score(P.sr, -m0), 0.0)

print(f"{'3  forecast used as detector (circular)':45s} AUC {SR['3  forecast used as detector (circular)'][0]:.3f}")
EV["sr_auc"] = SR; EV["sr_rate"] = float(P.sr.mean())


# what separates them? (rates by simple pre-treatment factors; association, not causation)
def rate_by(col, bins=None, labels=None):
    s = P[col] if bins is None else pd.cut(P[col], bins, labels=labels)
    return P.groupby(s).sr.agg(rate="mean", n="size").round(3)

print("\nsuper-responder rate by line number:\n", rate_by("line", [0, 1, 2, 9], ["1st line", "2nd line", "3rd+"]).to_string())

print("\nby years from diagnosis to biologic start (non-prevalent):\n", rate_by("dx_to_start_days", [-1, 365, 730, 1095, 20000], ["<1y", "1-2y", "2-3y", ">3y"]).to_string())

print("\nby Hurley:\n", rate_by("hurley").to_string()); print("\nby age at diagnosis tercile:\n", P.groupby(pd.qcut(P.age_dx, 3)).sr.agg(rate="mean", n="size").round(3).to_string())
tr_i, te_i = next(iter(StratifiedKFold(5, shuffle=True, random_state=0).split(P, P.sr)))

clf = HGBC(max_iter=200, learning_rate=.05, max_leaf_nodes=8, min_samples_leaf=40, random_state=0).fit(P.iloc[tr_i][pre_c], P.sr.iloc[tr_i])
from sklearn.inspection import permutation_importance
pi = permutation_importance(clf, P.iloc[te_i][pre_c], P.sr.iloc[te_i], scoring="roc_auc", n_repeats=5, random_state=0)
imp = pd.Series(pi.importances_mean, pre_c).sort_values(ascending=False).head(8); EV["sr_imp"] = imp; print("\npermutation importance (AUC drop) of the pre-treatment model:\n", imp.round(4).to_string())

check("V11 super-responders", "pre-treatment AUC is clearly below the circular reference (so we are not fooled by circularity)", SR["1b + pre-treatment history (at line start)"][0] < SR["3  forecast used as detector (circular)"][0] - 0.1)

8548 biologic patients with a target | proxy super-responders: 11.8% (challenge brief: ~1 in 8 = 12.5%)  | by Hurley: {2.0: 0.168, 3.0: 0.022}
1a demography + Hurley                        AUC 0.691 +- 0.002
1b + pre-treatment history (at line start)    AUC 0.756 +- 0.001
2  + first 6 months on drug                   AUC 0.881 +- 0.001
3  forecast used as detector (circular)       AUC 0.955

super-responder rate by line number:
            rate     n
line                 
1st line  0.143  6762
2nd line  0.030  1436
3rd+      0.003   350

by years from diagnosis to biologic start (non-prevalent):
                    rate     n
dx_to_start_days             
<1y               0.160  4180
1-2y              0.106  2231
2-3y              0.058   985
>3y               0.013   713

by Hurley:
          rate     n
hurley             
2.0     0.168  5602
3.0     0.022  2946

by age at diagnosis tercile:
                  rate     n
age_dx                     
(14.999, 25.0]  0.134  2799
(25.0, 3

# 7 - embeddings
The benchmark is not a late step on "complete patients". The forecast `M0` is built for **everyone** from clinical data. Embeddings then ask a narrower question: *for the patients who have one, does the embedding explain what `M0` got wrong?*

* The correction model learns the **out-of-fold residual** of `M0` (`target - M0 prediction`): `PCA(embedding)` -> ridge regression. Because it predicts the *leftover*, it can only add what the clinical data and visits did not already explain. (Using out-of-fold residuals matters: in-sample residuals of a boosted model are artificially small.)
* Everything is fitted inside the cross-validation fold; gains are **paired bootstrap** CIs over patients.
* Known small limitation: the out-of-fold `M0` predictions of the *training* patients came from models that had seen the *test fold* patients. This is a second-order effect; the shuffled-embedding null below checks that it does not manufacture gains.

Path: coverage >> confounds (batch/device/date) >> incremental value >> volume >> subgroups.

In [33]:
# function helper -- general
def _exists(d, name):
    """True if a local file or a GitHub file exists."""
    p = _src(d, name)
    if str(p).startswith("http"):
        try:
            with urllib.request.urlopen(urllib.request.Request(p, method="HEAD")):
                return True
        except (urllib.error.HTTPError, urllib.error.URLError):
            return False
    return os.path.exists(p)

def load_emb(d):
    """{"rnaseq": df, "images": df}; each df = metadata + embedding columns, one row per sample."""
    out = {}
    for nm, idc, dc in [("rnaseq", "rnaseq_sample_id", "collection_date"),
                        ("images", "image_id", "capture_date")]:
        fe, fm = f"{nm}_embeddings.csv", f"{nm}_metadata.csv"
        if not (_exists(d, fe) and _exists(d, fm)):
            print(f"load_emb: {nm} files not found in {d}")
            continue
        meta = pd.read_csv(_src(d, fm))
        emb = pd.read_csv(_src(d, fe))
        if idc not in meta.columns or idc not in emb.columns:      # fall back to the first column
            print(f"load_emb: '{idc}' not found for {nm}, using '{meta.columns[0]}'")
            idc = meta.columns[0]
        if dc in meta.columns:
            meta[dc] = pd.to_datetime(meta[dc], errors="coerce")
        meta = meta.set_index(idc)
        emb = emb.set_index(idc)
        emb = emb[[c for c in emb.columns if c not in meta.columns]]   # avoid duplicate column names
        out[nm] = meta.join(emb, how="inner")
    return out

def mk_corr(k):
    return make_pipeline(StandardScaler(), PCA(k, random_state=0), RidgeCV(alphas=np.logspace(-1, 4, 12)))

In [34]:
# function helpers -- analysis

# coverage
def coverage_row(m, nm):
    Xs = X.reindex(m.patient_id.values) if m is not None else X
    idx = Xs.index
    return dict(set=nm, n=len(Xs), pct_of_cohort=len(Xs) / len(X),
                on_biologic=Xs.on_bio.mean(), first_line=(Xs.cur_line == 1).mean(),
                hurley3=(Xs.hurley == 3).mean(), started_lt_1y_ago=(Xs.days_on_bio < 365).mean(),
                mean_target=y.reindex(idx).mean(), mean_M0_residual=res_m0.reindex(idx).mean())

# confound analysis
def confound(m, dims, tech, datecol, nm):
    r = res_m0.reindex(m.patient_id.values).values; grp = m[tech].values
    kw = stats.kruskal(*[r[grp == g] for g in np.unique(grp) if (grp == g).sum() > 5])
    d = (m[datecol] - m[datecol].min()).dt.days.values; rho = stats.spearmanr(grp, d)
    acc = np.mean([ (make_pipeline(StandardScaler(), LogisticRegression(C=.05, max_iter=300)).fit(m[dims].values[tr], grp[tr]).predict(m[dims].values[te]) == grp[te]).mean()
                    for tr, te in KFold(5, shuffle=True, random_state=0).split(m)])
    maj = pd.Series(grp).value_counts(normalize=True).iloc[0]
    return dict(modality=nm, n=len(m), tech=tech, residual_vs_tech_p=kw.pvalue, tech_vs_date_rho=rho.statistic, tech_vs_date_p=rho.pvalue, embedding_identifies_tech_acc=acc, majority_class=maj)

# emebedding incremental value
def eval_emb(m, dims, k=16, groups=None, shuffle=False, seed=0, mask=None):
    idx = m.patient_id.values; E = m[dims].values.astype(float); r = res_m0.reindex(idx).values
    if shuffle: E = E[np.random.default_rng(seed).permutation(len(E))]
    spl = KFold(5, shuffle=True, random_state=seed).split(E) if groups is None else GroupKFold(5).split(E, groups=groups)
    c = np.zeros(len(E))
    for tr, te in spl: c[te] = mk_corr(k).fit(E[tr], r[tr]).predict(E[te])
    return pd.DataFrame({"e0": np.abs(r), "e1": np.abs(r - c), "c": c}, index=idx)

# embedding effect analysis
def volume_curve(m, dims, sizes, k=16, seed=0):
    idx = m.patient_id.values; E = m[dims].values.astype(float); r = res_m0.reindex(idx).values; out = {s: [] for s in sizes}; rng = np.random.default_rng(seed)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(E):
        for s in sizes:
            if s <= len(tr):
                sub = rng.choice(tr, s, replace=False); c = mk_corr(min(k, s - 2)).fit(E[sub], r[sub]).predict(E[te]); out[s].append(np.abs(r[te]).mean() - np.abs(r[te] - c).mean())
    return pd.Series({s: np.mean(v) for s, v in out.items() if v})

In [35]:
# load train embeddings data

EMB = load_emb(DATA_DIR)
if "rnaseq" not in EMB or "images" not in EMB:
    raise FileNotFoundError(f"Embedding files not found in {DATA_DIR}. Loaded: {list(EMB)}")
rna, img = EMB["rnaseq"], EMB["images"]
rna = rna[rna.patient_id.isin(y.index)]
img = img[img.patient_id.isin(y.index)]
rdims = [c for c in rna.columns if "_dim_" in c]
idims = [c for c in img.columns if "_dim_" in c]
print(f"RNAseq: {rna.shape} ({len(rdims)} dims) | images: {img.shape} ({len(idims)} dims)")

# out-of-fold residual of the forecast, aligned by patient (safe against ordering problems)
res_m0 = y.reindex(X.index) - pd.Series(np.asarray(OOF["M0 blend"]), index=X.index)

RNAseq: (600, 134) (128 dims) | images: (2800, 263) (256 dims)


In [36]:
# coverage
COV = pd.DataFrame([coverage_row(None, "whole train cohort"),
                    coverage_row(img, "has image"),
                    coverage_row(rna, "has RNAseq (all also have image)")]).set_index("set")
print(COV.round(3).T.to_string())
EV["emb_cov"] = COV

set                whole train cohort  has image  has RNAseq (all also have image)
n                           21000.000   2800.000                           600.000
pct_of_cohort                   1.000      0.133                             0.029
on_biologic                     0.407      0.700                             1.000
first_line                      0.322      0.531                             0.662
hurley3                         0.245      0.300                             0.315
started_lt_1y_ago               0.076      0.277                             0.992
mean_target                     8.320      9.125                             8.722
mean_M0_residual                0.250      0.177                            -0.223


In [37]:
# RNAseq timing and overlap check
CUR_ALL = cur_lines(Tc)    # every open biologic line at the cutoff
st = CUR_ALL.start_date.reindex(rna.patient_id.values).values
EV["rna_date_eq_start"] = float((rna.collection_date.values == st).mean())
EV["rna_in_image"] = float(rna.patient_id.isin(img.patient_id).mean())
print(f"\nRNAseq collected exactly on the start date of the current biologic line: "
      f"{EV['rna_date_eq_start']:.0%}  |  RNAseq patients that also have an image: {EV['rna_in_image']:.0%}")

check("V12 embeddings", "RNAseq is a pre-treatment sample (collected at line start, not after)",
      EV["rna_date_eq_start"] > 0.95, f"{EV['rna_date_eq_start']:.0%}")
check("V12 embeddings", "embedding capture dates are all <= cutoff",
      max(rna.collection_date.max(), img.capture_date.max()) <= END)


RNAseq collected exactly on the start date of the current biologic line: 100%  |  RNAseq patients that also have an image: 100%
PASS | V12 embeddings | RNAseq is a pre-treatment sample (collected at line start, not after) | 100%
PASS | V12 embeddings | embedding capture dates are all <= cutoff | 


7b. Confound tests: batch / device / date (before believing any embedding effect)
Technical variation is the classic way an embedding "predicts" an outcome without biology: if a batch or scanner is also tied to a calendar period or a sub-population, the model can read the batch instead of the disease. We test: (i) does batch/device alone predict the **residual** of `M0`? (ii) does it track **collection date**? (iii) can the embedding itself **identify** the batch/device (so it carries a technical fingerprint)?

In [38]:
# test cofound

CF = pd.DataFrame([confound(rna, rdims, "batch_id", "collection_date", "RNAseq"), confound(img, idims, "device_id", "capture_date", "images")]).set_index("modality")
print(CF.round(4).T.to_string()); EV["conf"] = CF

check("V12 embeddings", "batch/device alone does not predict the M0 residual (p > 0.05)", bool((CF.residual_vs_tech_p > 0.05).all()), CF.residual_vs_tech_p.round(3).to_dict())

modality                         RNAseq     images
n                                   600       2800
tech                           batch_id  device_id
residual_vs_tech_p               0.4145     0.3667
tech_vs_date_rho                 0.2895     0.2876
tech_vs_date_p                      0.0        0.0
embedding_identifies_tech_acc    0.6383     0.5432
majority_class                   0.3433     0.1536
PASS | V12 embeddings | batch/device alone does not predict the M0 residual (p > 0.05) | {'RNAseq': 0.415, 'images': 0.367}


7c. Incremental value on top of `M0` (same patients, paired CI), with null and robustness tests
Rows: the correction fitted by 5-fold CV; **shuffled-embedding null** (rows permuted among patients: must give ~0); **leave-one-batch-out** / **leave-one-device-out** (the correction must work on a batch/device it never saw). If the gain survives these, it is not just a batch fingerprint.

In [39]:
# incremental value
rows = []; EVE = {}
for nm, m, dims, tech in [("RNAseq (n=%d)" % len(rna), rna, rdims, "batch_id"), ("images (n=%d)" % len(img), img, idims, "device_id")]:
    for tag, kw in [("5-fold CV", {}), ("shuffled embeddings (null)", dict(shuffle=True)), (f"leave-one-{tech.split('_')[0]}-out", dict(groups=m[tech].values))]:
        d = eval_emb(m, dims, **kw); g = paired_gain(d.e0.values, d.e1.values, n=500); EVE[(nm, tag)] = d
        rows.append(dict(modality=nm, test=tag, MAE_M0=d.e0.mean(), MAE_with_embedding=d.e1.mean(), gain=g["gain"], lo=g["lo"], hi=g["hi"]))

EM = pd.DataFrame(rows).set_index(["modality", "test"]); print(EM.round(4).to_string()); EV["emb_tab"] = EM

r_cv = EM.xs("5-fold CV", level=1); r_nl = EM.xs("shuffled embeddings (null)", level=1)
check("V12 embeddings", "shuffled-embedding null produces no spurious gain (upper CI < 0.03)", bool((r_nl.hi < 0.03).all()), r_nl.gain.round(3).to_dict())

rn_key = [k for k in EVE if k[0].startswith("RNAseq") and k[1].startswith("leave")][0]; gb = paired_gain(EVE[rn_key].e0.values, EVE[rn_key].e1.values, n=500)
check("V12 embeddings", "RNAseq gain survives leave-one-batch-out (CI above 0)", gb["lo"] > 0, fmt(gb))

                                            MAE_M0  MAE_with_embedding    gain      lo      hi
modality        test                                                                          
RNAseq (n=600)  5-fold CV                   2.0792              1.7065  0.3727  0.2799  0.4582
                shuffled embeddings (null)  2.0792              2.0836 -0.0044 -0.0232  0.0158
                leave-one-batch-out         2.0792              1.7269  0.3523  0.2514  0.4466
images (n=2800) 5-fold CV                   1.5026              1.5025  0.0001 -0.0130  0.0145
                shuffled embeddings (null)  1.5026              1.5147 -0.0121 -0.0186 -0.0053
                leave-one-device-out        1.5026              1.5034 -0.0008 -0.0135  0.0138
PASS | V12 embeddings | shuffled-embedding null produces no spurious gain (upper CI < 0.03) | {'RNAseq (n=600)': -0.004, 'images (n=2800)': -0.012}
PASS | V12 embeddings | RNAseq gain survives leave-one-batch-out (CI above 0) | +0.3523 (95%

7d. Where does it help, how much data does it need, and the routing decision
* **Image vs RNAseq on the same 600 patients**: is the image adding anything once RNAseq is present?
* **Image corrector on image-only patients** (those *without* RNAseq), split by biologic status.
* **Volume curve**: gain as a function of training samples (same 5 outer folds; the test part is fixed).
* **Subgroups for RNAseq**: first-line vs later lines, recent starters, Hurley stage.

In [40]:
# embedding efect analysis
d_r = EVE[[k for k in EVE if k[0].startswith("RNAseq") and k[1] == "5-fold CV"][0]]; d_i = EVE[[k for k in EVE if k[0].startswith("images") and k[1] == "5-fold CV"][0]]
both = d_r.index.intersection(d_i.index)
g_rna_vs_img = paired_gain(d_i.loc[both].e1.values, d_r.loc[both].e1.values, n=500)
print("on the RNAseq patients: RNAseq correction vs IMAGE correction (positive = RNAseq better):", fmt(g_rna_vs_img))

only_img = d_i.index.difference(rna.patient_id); onb = X.on_bio.reindex(only_img).values == 1
g_img_bio = paired_gain(d_i.loc[only_img].e0.values[onb], d_i.loc[only_img].e1.values[onb], n=500); g_img_non = paired_gain(d_i.loc[only_img].e0.values[~onb], d_i.loc[only_img].e1.values[~onb], n=500)
print(f"image correction, image-only patients on biologic ({onb.sum()}):", fmt(g_img_bio)); print(f"image correction, image-only patients NOT on biologic ({(~onb).sum()}):", fmt(g_img_non))

EV["g_rna_vs_img"], EV["g_img_bio"], EV["g_img_non"] = g_rna_vs_img, g_img_bio, g_img_non
ROUTE = dict(rnaseq=bool(r_cv.loc[[i for i in r_cv.index if i.startswith("RNAseq")][0], "lo"] > 0), images=bool(g_img_bio["lo"] > 0))
print("ROUTE (decided by CI > 0):", ROUTE)

VC = pd.DataFrame({"RNAseq": volume_curve(rna, rdims, [50, 100, 200, 400]), "images": volume_curve(img, idims, [100, 200, 500, 1000, 2200])}); print("\nMAE gain vs number of training samples:\n", VC.round(3).to_string()); EV["vol"] = VC

# subgroups for RNAseq
Xs = X.reindex(d_r.index); sg = {"first line": Xs.cur_line == 1, "later lines": Xs.cur_line > 1, "started < 1y ago": Xs.days_on_bio < 365, "started >= 1y ago": Xs.days_on_bio >= 365, "Hurley 2": Xs.hurley == 2, "Hurley 3": Xs.hurley == 3}
rows = []
for k_, mk_ in sg.items():
    mk_ = mk_.values; g = paired_gain(d_r.e0.values[mk_], d_r.e1.values[mk_], n=400); rows.append(dict(subgroup=k_, n=int(mk_.sum()), gain=g["gain"], lo=g["lo"], hi=g["hi"]))
SGR = pd.DataFrame(rows).set_index("subgroup"); print("\nRNAseq correction by subgroup:\n", SGR.round(3).to_string()); EV["rna_sub"] = SGR

on the RNAseq patients: RNAseq correction vs IMAGE correction (positive = RNAseq better): +0.2612 (95% CI +0.1699 .. +0.3413)
image correction, image-only patients on biologic (1360): -0.0314 (95% CI -0.0504 .. -0.0119)
image correction, image-only patients NOT on biologic (840): -0.0285 (95% CI -0.0537 .. +0.0013)
ROUTE (decided by CI > 0): {'rnaseq': True, 'images': False}

MAE gain vs number of training samples:
       RNAseq  images
50     0.328     NaN
100    0.329  -0.029
200    0.361  -0.013
400    0.367     NaN
500      NaN  -0.010
1000     NaN  -0.001
2200     NaN   0.000

RNAseq correction by subgroup:
                      n   gain     lo     hi
subgroup                                   
first line         397  0.462  0.326  0.613
later lines        203  0.197  0.055  0.340
started < 1y ago   595  0.377  0.283  0.478
started >= 1y ago    5 -0.191 -0.608  0.143
Hurley 2           411  0.387  0.262  0.503
Hurley 3           189  0.341  0.170  0.492


7e. Embeddings and super-responders (2b)
RNAseq was drawn **at the start of the current biologic line**, i.e. *before* treatment, so it is legitimately available "before treatment". Question: does it add to the pre-treatment clinical detector (7c, level 1b)? Small sample (the 600 RNAseq patients contain only a few dozen positives), so we repeat the CV 5 times and report the spread; treat it as a *lead*, not a conclusion.

In [41]:
# embedding in super responder effect
rr = rna.set_index("patient_id").join(P[["sr", "p_pre"]], how="inner").dropna(subset=["sr", "p_pre"]); Er = rr[rdims].values.astype(float); yr = rr.sr.values.astype(int)
lg = np.log(np.clip(rr.p_pre.values, 1e-4, 1 - 1e-4) / (1 - np.clip(rr.p_pre.values, 1e-4, 1 - 1e-4)))
o1m, o2m = np.zeros(len(rr)), np.zeros(len(rr)); R = 5

for s in range(R):
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=s).split(Er, yr):
        pca = make_pipeline(StandardScaler(), PCA(10, random_state=0)).fit(Er[tr]); Z = np.c_[lg, pca.transform(Er)]
        o2m[te] += LogisticRegression(C=.3, max_iter=500).fit(Z[tr], yr[tr]).predict_proba(Z[te])[:, 1] / R; o1m[te] += lg[te] / R

bs = np.random.default_rng(0).integers(0, len(rr), (500, len(rr)))
dA = np.array([roc_auc_score(yr[b], o2m[b]) - roc_auc_score(yr[b], o1m[b]) for b in bs if yr[b].min() != yr[b].max()])
EV["sr_rna"] = dict(n=len(rr), pos=int(yr.sum()), auc_tab=roc_auc_score(yr, o1m), auc_both=roc_auc_score(yr, o2m), lo=float(np.percentile(dA, 2.5)), hi=float(np.percentile(dA, 97.5)))

print(f"RNAseq patients with a super-responder label: {len(rr)} (positives {yr.sum()})  |  AUC pre-treatment clinical only {EV['sr_rna']['auc_tab']:.3f}  ->  + RNAseq {EV['sr_rna']['auc_both']:.3f}  (bootstrap 95% CI of the AUC gain {EV['sr_rna']['lo']:+.3f} .. {EV['sr_rna']['hi']:+.3f})")

RNAseq patients with a super-responder label: 600 (positives 49)  |  AUC pre-treatment clinical only 0.782  ->  + RNAseq 0.944  (bootstrap 95% CI of the AUC gain +0.108 .. +0.225)


## 8. Signal labelling, decide which to use, analyse the signal reability. 
(1) how much the model **relies on each feature family** (grouped permutation importance: shuffle a whole family at once in held-out patients and see how much MAE rises), each family carrying a label;
(2) a **findings ledger**: every pattern we relied on or tested, with the *measured* evidence, its label, and what it means for the model.

How the labels were decided in this notebook (rules, so you can challenge them):
* **CLINICAL**: persists *within* patient or *within* site, is consistent across independent measures (e.g. IHS4 and surgeries) or survives held-out-group validation, and has a mechanism.
* **ARTEFACT**: varies with who records / how it is recorded far beyond chance, or is a placeholder/construction effect, or encodes a batch/device.
* **UNDETERMINED**: the data cannot separate the two. The ledger states **what evidence would settle it**.

In [42]:
# function helper -- analysis
def mdiff(a, b, n=1000, seed=0):
    r = np.random.default_rng(seed); a, b = np.asarray(a, float), np.asarray(b, float); a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    m = np.array([r.choice(a, len(a)).mean() - r.choice(b, len(b)).mean() for _ in range(n)]); return a.mean() - b.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

In [43]:
# 8a. reliance of the model on each feature family (grouped permutation importance on held-out folds)
fam_f = {g: [c for c in FEATS if fam[c] == g] for g in sorted(set(fam[FEATS]))}
rows = {g: [] for g in fam_f}; rng = np.random.default_rng(0)

for tr, te in SPL[:3]:
    m_ = mk().fit(X.iloc[tr][FEATS], y.iloc[tr]); Xt = X.iloc[te][FEATS]; base_m = mae(y.iloc[te], m_.predict(Xt))
    for g, cols in fam_f.items():
        d = []
        for _ in range(3):
            Z = Xt.copy(); perm = rng.permutation(len(Z)); Z[cols] = Xt[cols].values[perm]; d.append(mae(y.iloc[te], m_.predict(Z)) - base_m)
        rows[g].append(np.mean(d))

REL = pd.Series({g: np.mean(v) for g, v in rows.items()}).sort_values(ascending=False)
LAB = {"hurley": "CLINICAL", "biologic": "CLINICAL", "surgeries": "CLINICAL", "antibiotics": "CLINICAL", "visit_scores": "CLINICAL (level) with sampling bias",
       "visit_behaviour": "UNDETERMINED (disease activity + site habit)", "hidden_flare": "UNDETERMINED (real unseen flares + documentation)", "demographics": "mixed: small CLINICAL effects + prevalent-cohort ARTEFACT", "practitioner": "ARTEFACT (excluded)"}
RELT = pd.DataFrame({"MAE increase when family shuffled": REL.round(4), "label": [LAB.get(g, "") for g in REL.index], "n_features": [len(fam_f[g]) for g in REL.index]}); print(RELT.to_string()); EV["rel"] = RELT

                 MAE increase when family shuffled                                                      label  n_features
visit_scores                                2.8360                        CLINICAL (level) with sampling bias          36
hurley                                      1.0608                                                   CLINICAL           1
biologic                                    0.2374                                                   CLINICAL           7
antibiotics                                 0.0528                                                   CLINICAL           5
hidden_flare                                0.0505          UNDETERMINED (real unseen flares + documentation)           9
visit_behaviour                             0.0474               UNDETERMINED (disease activity + site habit)          18
surgeries                                   0.0265                                                   CLINICAL           5
demographics            

In [44]:
# 8b. evidence for the findings ledger.

adj = pd.Series(y.values - B["B2 Hurley x biologic median"].values, index=X.index)         # target minus its Hurley x biologic median: "adjusted target"
F_ = {}

# F1 Hurley
mh = y.groupby(X.hurley).mean().round(2).to_dict(); sh_ = X.groupby("hurley").surg_365d.mean().round(2).to_dict()
F_["F1"] = (f"Hurley stage drives severity: mean target {mh}; surgeries in last year {sh_}", "CLINICAL")

# F2 biologic
d2 = {int(h): round(float(y[(X.hurley == h) & (X.on_bio == 1)].mean() - y[(X.hurley == h) & (X.on_bio == 0)].mean()), 2) for h in (2, 3)}
F_["F2"] = (f"On a biologic the target is lower within the same Hurley stage: difference on - off = {d2}", "CLINICAL")

# F3 stopped biologic looks like never treated
st_ = (X.on_bio == 0) & (X.n_lines > 0); nv = (X.n_lines == 0) & (X.hurley >= 2)
d3 = {int(h): (round(float(y[st_ & (X.hurley == h)].mean()), 2), round(float(y[nv & (X.hurley == h)].mean()), 2)) for h in (2, 3)}
F_["F3"] = (f"Patients who stopped a biologic (mean target) vs never-treated, by Hurley (stopped, never): {d3}. They are back near untreated level.", "CLINICAL")

# F4a / F4b flare visits
fs = pd.qcut(X.v_flare_share_365, 3, duplicates="drop"); ov = (X.s_mean_365 - y).groupby(fs).mean().round(2)
F_["F4a"] = (f"Within the same patient, scores at flare visits are {EV['within_patient_flare_gap']:+.2f} points above background visits", "CLINICAL")
F_["F4b"] = (f"Raw 365-day mean score minus target by tercile of flare-visit share: {ov.round(2).tolist()} (overstates the level more when most visits are flare visits)", "ARTEFACT")

# F5 practitioner scoring habit
gp_ = EV["abl"].loc["C + practitioner features"]
F_["F5"] = (f"Practitioner scoring rates vary {EV['score_rate_z_sd']:.1f}x more than chance (z-score sd; chance = 1). Practitioner features' MAE gain {gp_.gain_vs_base:+.4f} (CI {gp_.ci_lo:+.4f}..{gp_.ci_hi:+.4f}); with practitioners held out {fmt(EV['prac_gain_heldout'])}", "ARTEFACT")

# F6 practitioner score offset (do sites score systematically high/low relative to the underlying level?)
dd = (X.s_mean_365 - y).dropna(); pidx = X.prac_num.reindex(dd.index); pm = dd.groupby(pidx).agg(["mean", "size"]); pm = pm[pm["size"] >= 10]
v_obs = pm["mean"].var(); rs_ = np.random.default_rng(0); perm_v = []
for _ in range(300):
    sh2 = dd.groupby(rs_.permutation(pidx.values)).agg(["mean", "size"]); perm_v.append(sh2[sh2["size"] >= 10]["mean"].var())
p6 = float((np.array(perm_v) >= v_obs).mean()); ratio6 = float(v_obs / np.mean(perm_v))
F_["F6"] = (f"Site-level offset of observed scores vs the underlying level (score minus target): variance of practitioner means is {ratio6:.2f}x the shuffled-site expectation (permutation p = {p6:.3f}, {len(pm)} practitioners). Statistically real but small. Disease severity cannot differ between a score and its own target, so a site offset in this difference is a scoring habit", "ARTEFACT" if p6 < 0.01 else "UNDETERMINED")

# F7 informative visiting
def within_rho(a, b, g):
    ad, bd = a - a.groupby(g).transform("mean"), b - b.groupby(g).transform("mean"); m = ad.notna() & bd.notna(); return stats.spearmanr(ad[m], bd[m]).statistic
hb_key = X.hurley.astype(str) + "_" + X.on_bio.astype(str); r_all = within_rho(X.v_n_365, adj, hb_key); r_site = within_rho(X.v_n_365, adj, hb_key + "_" + X.prac_num.astype(str))
g7 = EV["abl"].loc["C - visit behaviour (no visit counts / flare share / gaps)"]
F_["F7"] = (f"Visits in last year vs adjusted target: Spearman {r_all:+.2f} within Hurley x biologic, {r_site:+.2f} also within the same practitioner. Dropping visit-behaviour features changes MAE by {-g7.gain_vs_base:+.4f}", "CLINICAL" if (abs(r_site) > 0.7 * abs(r_all) and abs(r_all) > 0.1) else "UNDETERMINED")

# F8 no scored visit
ns = X.s_mean_365.isna(); d8 = mdiff(adj[ns], adj[~ns]); d8lo = mdiff(adj[ns & (terc == "low documentation")], adj[~ns & (terc == "low documentation")]); d8hi = mdiff(adj[ns & (terc == "high documentation")], adj[~ns & (terc == "high documentation")])
same8 = (d8[2] < 0) and (d8lo[2] < 0) and (d8hi[2] < 0) and abs(d8lo[0] - d8hi[0]) < 0.3 * abs(d8[0])
F_["F8"] = (f"Patients with no scored visit in the last year ({int(ns.sum())}): adjusted target {d8[0]:+.2f} (CI {d8[1]:+.2f}..{d8[2]:+.2f}); in low-documentation sites {d8lo[0]:+.2f}, in high-documentation sites {d8hi[0]:+.2f}. If poor documentation were the cause the gap would be larger in low-documentation sites; it is the same", "CLINICAL" if same8 else "UNDETERMINED")

# F9 prevalent cohort
pv = X.prevalent == 1; d9 = mdiff(adj[pv], adj[~pv])
F_["F9a"] = (f"{int(pv.sum())} patients have the placeholder diagnosis date 2016-01-02, so their age at diagnosis, years since diagnosis and time to biologic are wrong", "ARTEFACT")
F_["F9b"] = (f"Prevalent-cohort patients have an adjusted target {d9[0]:+.2f} (CI {d9[1]:+.2f}..{d9[2]:+.2f}) vs the rest", "UNDETERMINED")

# F10 surgery documentation drift
s_ = T["sur"].assign(yr=T["sur"].procedure_date.dt.year, has=T["sur"].visit_id.notna()); drift = s_.groupby("yr").has.mean(); 
F_["F10"] = (f"Share of surgeries carrying a visit_id: {drift.loc[2016]:.0%} in 2016 -> {drift.loc[2025]:.0%} in 2025 (surgery has no biological reason to lose its visit link)", "ARTEFACT")

# F11 calendar year of diagnosis
dxy = dem_.diagnosis_date.dt.year.where(~pv); m11 = adj.groupby(dxy).mean().dropna(); sl = stats.linregress(m11.index.values.astype(float), m11.values)
F_["F11"] = (f"Adjusted target vs calendar year of diagnosis (non-prevalent): slope {sl.slope:+.3f} per year (p = {sl.pvalue:.2g}); diagnoses enter the data in blocks (only a handful in 2024)", "UNDETERMINED" if sl.pvalue < 0.05 else "UNDETERMINED")

# F12 sex
d12 = mdiff(adj[X.sex == 1], adj[X.sex == 0]); d12s = mdiff(X.surg_365d[X.sex == 1], X.surg_365d[X.sex == 0])
F_["F12"] = (f"Male vs female: adjusted target {d12[0]:+.2f} (CI {d12[1]:+.2f}..{d12[2]:+.2f}); surgeries in last year {d12s[0]:+.2f} (CI {d12s[1]:+.2f}..{d12s[2]:+.2f})", "CLINICAL" if (d12[2] < 0 and d12s[2] < 0) else "UNDETERMINED")

# F13 missing demographics
d13 = {c: mdiff(adj[X[c].isna()], adj[X[c].notna()]) for c in ["bmi", "smoking_status", "ethnicity", "family_history_hs"]}
F_["F13"] = ("Adjusted target when a demographic field is missing vs present: " + "; ".join(f"{c} {v[0]:+.2f} ({v[1]:+.2f}..{v[2]:+.2f})" for c, v in d13.items()), "UNDETERMINED")

# F14 antibiotic start sits on a flare
F_["F14"] = (f"Scores inside the first 14 days of an antibiotic course exceed the same patient's other scores by (points, by Hurley stage) { {int(k): round(v, 2) for k, v in UPLIFT.items()} }", "CLINICAL")

# F15 antibiotic / surgery history redundant once visits are known
for nm, cols in {"drop antibiotics family": [c for c in FEATS if fam[c] != "antibiotics"], "drop surgeries family": [c for c in FEATS if fam[c] != "surgeries"]}.items():
    OOF[nm] = oof(cols); ERR[nm] = np.abs(y.values - OOF[nm]); EV[nm] = paired_gain(ERR[nm], ERR["FINAL single"])
F_["F15"] = (f"Removing the antibiotic-count family makes MAE worse by {EV['drop antibiotics family']['gain']:+.4f} (CI {EV['drop antibiotics family']['lo']:+.4f}..{EV['drop antibiotics family']['hi']:+.4f}); removing the surgery family {EV['drop surgeries family']['gain']:+.4f} (CI {EV['drop surgeries family']['lo']:+.4f}..{EV['drop surgeries family']['hi']:+.4f}). Antibiotic counts add a small real gain; surgeries are redundant once visits and antibiotics are known", "CLINICAL")

# F16 hidden flare
F_["F16"] = (f"Hidden-flare features: MAE gain {fmt(paired_gain(ERR['C  base (tabular + visits)'], ERR['C + hidden flare']))}; gain by documentation tercile {EV['hf_terc'].gain.round(4).to_dict()}; rho(site scoring rate, base-model residual) = {EV['prac_rate_res_rho']:+.2f}; additive rule does not beat the plain shrunk level (MAE by weight on expected hidden burden: { {k: round(v, 3) for k, v in EV['additive_mae'].items()} }). The gain is the same in every documentation tercile and site residuals show no documentation trend: this leans clinical, but a documentation proxy is not excluded", "UNDETERMINED")

# F17 response types
F_["F17"] = (f"Response types (deep/partial/minimal = {({k: round(v,2) for k,v in EV['resp_shares'].items()})}): split-half r = {EV['resp_split_half']:.2f}; deep-vs-minimal target gap in Hurley 2 = {EV['resp_target_gap']:.1f}; forecast gate {fmt(EV['resp_gate_all'])}", "CLINICAL")

# F18 time from diagnosis to first biologic / F19 line number (super-responder proxy)
r18 = P.groupby(pd.cut(P.dx_to_start_days, [-1, 365, 730, 1095, 20000], labels=["<1y", "1-2y", "2-3y", ">3y"])).sr.mean().round(3).to_dict()
r19 = P.groupby(pd.cut(P.line, [0, 1, 2, 9], labels=["1st", "2nd", "3rd+"])).sr.mean().round(3).to_dict()
F_["F18"] = (f"Super-responder rate by years from diagnosis to start: {r18}", "UNDETERMINED")
F_["F19"] = (f"Super-responder rate by line number: {r19}", "CLINICAL")

# F20-22 embeddings
EMt = EV["emb_tab"]; rk = [i for i in EMt.index if i[0].startswith("RNAseq") and i[1].startswith("leave")][0]; rc = [i for i in EMt.index if i[0].startswith("RNAseq") and i[1] == "5-fold CV"][0]; ic = [i for i in EMt.index if i[0].startswith("images") and i[1] == "5-fold CV"][0]
F_["F20"] = (f"RNAseq correction on {len(rna)} biologic patients: gain {EMt.loc[rc,'gain']:+.3f} (CI {EMt.loc[rc,'lo']:+.3f}..{EMt.loc[rc,'hi']:+.3f}); leave-one-batch-out {EMt.loc[rk,'gain']:+.3f} (CI {EMt.loc[rk,'lo']:+.3f}..{EMt.loc[rk,'hi']:+.3f}); shuffled null ~0", "CLINICAL" if EMt.loc[rk, "lo"] > 0 else "UNDETERMINED")
F_["F21"] = (f"Embeddings identify the batch/device (accuracy {EV['conf'].embedding_identifies_tech_acc.round(2).to_dict()} vs majority {EV['conf'].majority_class.round(2).to_dict()}); batch tracks collection date (rho {EV['conf'].tech_vs_date_rho.round(2).to_dict()})", "ARTEFACT")
F_["F22"] = (f"Image correction: gain {EMt.loc[ic,'gain']:+.3f} (CI {EMt.loc[ic,'lo']:+.3f}..{EMt.loc[ic,'hi']:+.3f}) on {len(img)} patients; on biologic image-only patients {fmt(EV['g_img_bio'])}; images add little once RNAseq is present ({fmt(EV['g_rna_vs_img'])} RNAseq vs image on the same patients)", "UNDETERMINED")

# F23 dropouts
sl_ = EV["slices"].loc["days since last visit"].MAE; F_["F23"] = (f"MAE by time since last visit ranges only {sl_.min():.2f}..{sl_.max():.2f}: long absence from clinic does not make the forecast much worse here", "UNDETERMINED")

META = {
 "F1": ("Hurley stage / surgeries as the dominant axis", "Model uses it. Carries to test_2 (stage is a clinical property).", ""),
 "F2": ("Biologic lowers the target within stage", "Model uses it; basis for Exercise 2a routing.", ""),
 "F3": ("After stopping, activity returns to untreated level", "Model uses days_since_bio_stop. Main worsening window.", ""),
 "F4a": ("Higher scores at flare visits within the same patient", "Visit-type and flare-share features let the model separate flare from baseline.", ""),
 "F4b": ("Raw recent-score average overstates the usual level for flare-heavy patients", "Never carry the raw average forward (baseline B3/B5 are worse than B2); use non-flare, decayed, shrunk levels.", ""),
 "F5": ("Practitioner scoring rate", "Excluded from the forecast; checked leave-practitioner-out.", ""),
 "F6": ("Practitioner score offset vs underlying level", "Not modelled as a feature. If test_2 has other practitioners, an offset model would break.", ""),
 "F7": ("Visits per year (informative visiting)", "Kept (ablation shows it helps); watch it on test_2 where visiting habits may differ.", "A test where visit rate is assigned independently of disease (e.g. scheduled-only sites) would settle the remaining site-habit share."),
 "F8": ("Patients with no scored visit have a milder course", "Handled natively (missing features, falls back to tabular signal). Do not impute. Probably milder disease -> fewer scored visits.", "Cannot exclude a patient-level habit (patients who skip scoring); a few patients with an independently measured IHS4 who were never scored would settle it."),
 "F9a": ("Placeholder diagnosis date", "Dependent features set to missing for the prevalent cohort.", ""),
 "F9b": ("Prevalent cohort higher target", "Flag kept (small, possibly real longer disease course), but could be censoring.", "True onset dates for the prevalent patients; or test_2 composition of this group."),
 "F10": ("Surgery visit_id drift", "Hidden-flare block matches surgeries to visits by date, not by visit_id.", ""),
 "F11": ("Calendar year of diagnosis", "Raw dx year is not a feature. Era effects can be a recording change (documentation) or a real cohort change.", "Same-year patients from two sites with known recording practice, or years 2024-2025 in test_2."),
 "F12": ("Sex", "Small effect, in the model through the demographics family.", ""),
 "F13": ("Missingness of demographics", "No explicit flags; GBM sees the NaNs. Effect not distinguishable from zero.", "A larger sample or documented reason for missing fields."),
 "F14": ("Antibiotic course marks a flare", "Basis of the hidden-flare block.", ""),
 "F15": ("Antibiotic counts add a small gain; rich medication history does not", "Antibiotic counts kept. No extra medication-history features (your earlier negative result: drug mix, burstiness, biologic history added nothing).", ""),
 "F16": ("Hidden-flare features", "Kept: small but significant gain. The mix of real unseen flares and documentation cannot be separated here.", "Patients with an independent flare log (patient diary, photos) to compare against; or sites with matched antibiotic counts but very different documentation."),
 "F17": ("Response types", "Explanation + 2b only. Not a forecast input (gate).", ""),
 "F18": ("Longer wait to first biologic and lower response", "Report as an association; confounded by indication and era.", "Randomised or quasi-random start timing (e.g. waiting lists)."),
 "F19": ("First-line biologic = more super-responders", "Report as 2b factor; usable before/at start of treatment.", ""),
 "F20": ("RNAseq adds to the forecast for biologic patients", "Used only for patients with RNAseq. Scope: biologic patients, recent starters over-represented.", "Check on test_1/test_2 RNAseq patients."),
 "F21": ("Embeddings carry a batch/device fingerprint", "Handled by only trusting gains that survive leave-one-batch/device-out.", ""),
 "F22": ("Image embedding value", "Not used by default (routing rule). No detectable gain beyond M0.", "More image-labelled patients, or a test set where image patients differ."),
 "F23": ("Dropouts barely hurt", "Reported; re-check on test sets.", "Test_2 dropout mechanism."),
}

In [45]:
FIND = pd.DataFrame([dict(id=k, finding=META[k][0], label=F_[k][1], evidence=F_[k][0], action_in_model=META[k][1], what_would_settle=META[k][2]) for k in META]).set_index("id")
EV["find"] = FIND
print(FIND.label.value_counts().to_string())

label
CLINICAL        12
UNDETERMINED     7
ARTEFACT         6


## 9. Training data fuser >> `HSForecaster`
One object that holds everything fitted on training data:
* the **uplift constants** (fitted on train, then frozen),
* the **blend** of three gradient-boosting models (forecast core, `M0`),
* the **embedding correctors** (RNAseq, and images only if the routing rule allowed it), trained on **inner out-of-fold residuals** of `M0`,
* a small reference sample of training features used for the shift check.

`predict_tables(tables, emb)` takes raw test tables (same format as train) and returns a DataFrame `patient_id, predicted_ihs4`. Patients missing from any table still get a prediction (their features are missing, which the model handles).

In [46]:
# define HSForecaster
class HSForecaster:
    def __init__(self, cfg, route, uplift): self.cfg, self.route, self.uplift = dict(cfg), dict(route), dict(uplift); self.corr = {}
    def fit(self, Xt, yt, emb=None, folds=5, seed=0):
        self.cols = feature_cols(self.cfg); self.models = fit_blend(Xt, yt, self.cols)
        self.ref = Xt[self.cols].sample(min(5000, len(Xt)), random_state=0)
        emb = emb or {}
        if any(self.route.get(k) and k in emb for k in ("rnaseq", "images")):
            o = np.zeros(len(Xt))                                              # inner out-of-fold M0 predictions -> honest residuals
            for tr, te in KFold(folds, shuffle=True, random_state=seed).split(Xt): o[te] = predict_blend(fit_blend(Xt.iloc[tr], yt.iloc[tr], self.cols), Xt.iloc[te], self.cols)
            resid = pd.Series(yt.values - o, index=Xt.index)
            for key in ("rnaseq", "images"):
                if self.route.get(key) and key in emb:
                    m = emb[key]; m = m[m.patient_id.isin(Xt.index)]; dims = [c for c in m if "_dim_" in c]; dc = "collection_date" if key == "rnaseq" else "capture_date"
                    m = m.sort_values(dc).drop_duplicates("patient_id", keep="last")
                    self.corr[key] = (dims, mk_corr(16).fit(m[dims].values.astype(float), resid.reindex(m.patient_id).values))
        return self
    def predict_tables(self, Tt, emb=None, ids=None, shift_check=False, name="new set"):
        ids = pd.Index(sorted(set().union(*[set(Tt[k].patient_id) for k in ("dem", "asg", "trt", "sur", "vis")]))) if ids is None else pd.Index(ids)
        f = build(Tt, END, uplift=self.uplift, ids=ids)
        if shift_check: self.shift = shift_report(self.ref, f.reindex(columns=self.cols), self.cols, name=name); print(self.shift[1].round(3).to_string())
        base = predict_blend(self.models, f, self.cols); pred = pd.Series(base, index=ids); self.last_base = pred.copy(); used = {}
        emb = emb or {}
        for key in ("rnaseq", "images"):                                       # RNAseq first; image correction only for patients without RNAseq
            if key in self.corr and key in emb:
                dims, cm = self.corr[key]; m = emb[key]; dc = "collection_date" if key == "rnaseq" else "capture_date"
                m = m.sort_values(dc).drop_duplicates("patient_id", keep="last"); m = m[m.patient_id.isin(ids) & ~m.patient_id.isin(list(used))]
                if len(m): pred.loc[m.patient_id.values] += cm.predict(m[dims].values.astype(float)); used.update({p: key for p in m.patient_id})
        out = pd.DataFrame({"patient_id": ids, "predicted_ihs4": np.clip(pred.values, 0, None)}); self.last_routes = pd.Series(used, name="embedding_used")
        assert out.patient_id.is_unique and out.predicted_ihs4.notna().all() and len(out) == len(ids)
        return out
print("HSForecaster defined")

HSForecaster defined


In [47]:
# test the model on 20% patient
# randomize the patient
def subset(Tx, ids):
    ids = set(ids); return {k: (v[v.patient_id.isin(ids)] if k != "dem" else v[v.patient_id.isin(ids)]) for k, v in Tx.items()}
def subset_emb(E, ids): return {k: v[v.patient_id.isin(set(ids))] for k, v in E.items()}

perm = np.random.default_rng(42).permutation(y.index.values); te_ids, tr_ids = perm[:4200], perm[4200:]
T_tr, T_te = subset(T, tr_ids), subset(T, te_ids)
t0 = time.time(); f_tr = build(T_tr, END); f_tr = f_tr.loc[tr_ids]
dry = HSForecaster(CFG, ROUTE, f_tr.attrs["uplift"]).fit(f_tr, y.loc[tr_ids], emb=subset_emb(EMB, tr_ids)); print(f"fit in {time.time()-t0:.0f}s; correctors fitted for: {list(dry.corr)}")
pred = dry.predict_tables(T_te, emb=subset_emb(EMB, te_ids), shift_check=True, name="dry-run hold-out")

pr_ = pred.set_index("patient_id").predicted_ihs4; yt_ = y.loc[pr_.index]; base_ = dry.last_base
rna_te = EMB["rnaseq"][EMB["rnaseq"].patient_id.isin(te_ids)].patient_id.values
bz = (X.loc[pr_.index].on_bio == 1)
print(f"\nhold-out MAE (4,200 unseen patients): {mae(yt_, pr_):.4f}   [M0 only {mae(yt_, base_):.4f}]  | on biologic (2a): {mae(yt_[bz], pr_[bz]):.4f}")
print(f"RNAseq patients in hold-out ({len(rna_te)}): M0 {mae(yt_.loc[rna_te], base_.loc[rna_te]):.3f} -> with RNAseq correction {mae(yt_.loc[rna_te], pr_.loc[rna_te]):.3f}")

Xte = X.loc[pr_.index]; hbm = y.loc[tr_ids].groupby([X.hurley.loc[tr_ids], X.on_bio.loc[tr_ids]]).median()
b2 = pd.Series([hbm.get((h, b), y.median()) for h, b in zip(Xte.hurley, Xte.on_bio)], index=Xte.index)
print(f"baselines on the same patients: Hurley x biologic {mae(yt_, b2):.3f} | population median {mae(yt_, y.loc[tr_ids].median()):.3f}")

EV["dry"] = dict(mae=mae(yt_, pr_), mae_m0=mae(yt_, base_), mae_bio=mae(yt_[bz], pr_[bz]), b2=mae(yt_, b2))
check("V13 dry run", "hold-out MAE is in line with cross-validated MAE (within 0.15)", abs(EV["dry"]["mae_m0"] - EV["m0_mae"]) < 0.15, f"{EV['dry']['mae_m0']:.3f} vs CV {EV['m0_mae']:.3f}")
check("V13 dry run", "pipeline beats Hurley x biologic on unseen patients", EV["dry"]["mae"] < EV["dry"]["b2"])
check("V13 dry run", "one prediction per patient, no NaN, none negative", pred.patient_id.is_unique and pred.predicted_ihs4.notna().all() and (pred.predicted_ihs4 >= 0).all(), f"{len(pred)} rows")
pred.head()

fit in 89s; correctors fitted for: ['rnaseq']
[dry-run hold-out] adversarial AUC train-vs-new = 0.495  (0.5 = same population)
                         KS  missing_rate_gap
feature                                      
days_since_bio_stop   0.051            -0.004
s_ondrug_mean         0.044             0.006
s_nonflare_mean_90    0.038             0.015
days_on_bio           0.038             0.016
s_nonflare_mean_30    0.031             0.009
s_follow_up_365       0.030             0.006
s_flare_reactive_365  0.029             0.005
s_mean_90             0.029             0.012
v_flare_share_30      0.027             0.012
abx_all               0.027             0.000

hold-out MAE (4,200 unseen patients): 1.4053   [M0 only 1.4191]  | on biologic (2a): 1.1462
RNAseq patients in hold-out (113): M0 2.203 -> with RNAseq correction 1.688
baselines on the same patients: Hurley x biologic 2.302 | population median 4.403
PASS | V13 dry run | hold-out MAE is in line with cross-validated MAE 

,patient_id,predicted_ihs4
0,HS000004,18.065402
1,HS000026,9.825562
2,HS000027,5.632226
3,HS000044,6.667001
4,HS000071,13.143536


In [48]:
# try the model to all patient
t0 = time.time(); FINAL = HSForecaster(CFG, ROUTE, UPLIFT).fit(X, y, emb=EMB); print(f"production model fitted in {time.time()-t0:.0f}s | correctors: {list(FINAL.corr)} | features: {len(FINAL.cols)} | CFG: {CFG} | ROUTE: {ROUTE}")

production model fitted in 98s | correctors: ['rnaseq'] | features: 91 | CFG: {'use_practitioner': False, 'use_hidden_flare': True, 'use_response_types': False} | ROUTE: {'rnaseq': True, 'images': False}


In [49]:
# super responder classifier

class SRDetector:
    # P(super-responder) for patients on a biologic at the cutoff (Exercise 2b / the super_responder_prob column).
    # Label = the proxy of section 7c (target < 25% of the non-biologic mean target of the same Hurley stage). Fitted on TRAIN only.
    # Inputs = what is known at biologic start (7c level 1b) + the first 6 months on drug (level 2); RNAseq stacked on top only if it passed a CI gate.
    def __init__(self, nb_mean, cols, drugs): self.nb_mean, self.cols, self.drugs = nb_mean, list(cols), list(drugs)
    def feats(self, Tc, f):
        o = pretx_feats(Tc, f); cur = cur_lines(Tc)
        o["drug"] = cur.drug_name.map({d: float(i) for i, d in enumerate(self.drugs)}).reindex(o.index)   # frozen drug codes (train list), not recomputed on the test set
        return o.reindex(columns=self.cols)
    @staticmethod
    def _mk(): return CalibratedClassifierCV(HGBC(max_iter=200, learning_rate=.05, max_leaf_nodes=8, min_samples_leaf=40, random_state=0), method="isotonic", cv=5)
    @staticmethod
    def _logit(p): p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))
    def _latest_rna(self, emb):
        if not emb or "rnaseq" not in emb: return None
        return emb["rnaseq"].sort_values("collection_date").drop_duplicates("patient_id", keep="last").set_index("patient_id")
    def fit(self, Tc, f, y, emb=None, seed=0, reps=5):
        Z = self.feats(Tc, f); hz = pretx_feats(Tc, f, early=False).hurley
        Z = Z.join(y.rename("t"), how="inner"); lab = (Z.t < 0.25 * hz.reindex(Z.index).map(self.nb_mean)).astype(int); Z = Z[self.cols]
        self.rate = float(lab.mean()); self.base = self._mk().fit(Z, lab)
        oof = np.zeros(len(Z))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Z, lab): oof[te] = self._mk().fit(Z.iloc[tr], lab.iloc[tr]).predict_proba(Z.iloc[te])[:, 1]
        self.oof = pd.Series(oof, Z.index); self.lab = lab
        self.ev = dict(n=len(Z), pos=int(lab.sum()), rate=self.rate, auc=roc_auc_score(lab, oof), brier=float(np.mean((oof - lab) ** 2)), brier_const=float(self.rate * (1 - self.rate)), mean_oof=float(oof.mean()))
        sh = np.random.default_rng(seed).permutation(lab.values); o2 = np.zeros(len(Z))                                  # label-shuffle null
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Z, sh): o2[te] = HGBC(max_iter=100, learning_rate=.05, max_leaf_nodes=8, min_samples_leaf=40, random_state=0).fit(Z.iloc[tr], sh[tr]).predict_proba(Z.iloc[te])[:, 1]
        self.ev["auc_null"] = roc_auc_score(sh, o2)
        self.use_rna, self.rna_ev, self.stack = False, None, None
        r = self._latest_rna(emb)
        if r is not None:
            ids = r.index.intersection(Z.index); dims = [c for c in r if "_dim_" in c]; self.rdims = dims
            Er = r.loc[ids, dims].values.astype(float); yr = lab.loc[ids].values; lg = self._logit(self.oof.loc[ids].values)
            if len(ids) >= 100 and yr.sum() >= 20:
                o1, o2 = np.zeros(len(ids)), np.zeros(len(ids))
                for s in range(reps):
                    for tr, te in StratifiedKFold(5, shuffle=True, random_state=s).split(Er, yr):
                        pca = make_pipeline(StandardScaler(), PCA(10, random_state=0)).fit(Er[tr]); Zs = np.c_[lg, pca.transform(Er)]
                        o2[te] += LogisticRegression(C=.3, max_iter=500).fit(Zs[tr], yr[tr]).predict_proba(Zs[te])[:, 1] / reps; o1[te] += lg[te] / reps
                bs = np.random.default_rng(0).integers(0, len(ids), (500, len(ids)))
                dA = np.array([roc_auc_score(yr[b], o2[b]) - roc_auc_score(yr[b], o1[b]) for b in bs if yr[b].min() != yr[b].max()])
                self.rna_ev = dict(n=len(ids), pos=int(yr.sum()), auc_base=roc_auc_score(yr, o1), auc_stack=roc_auc_score(yr, o2), lo=float(np.percentile(dA, 2.5)), hi=float(np.percentile(dA, 97.5)))
                self.use_rna = self.rna_ev["lo"] > 0                                                                      # same gate rule as the forecast: CI of the gain must exclude 0
                if self.use_rna:
                    pca = make_pipeline(StandardScaler(), PCA(10, random_state=0)).fit(Er); Zs = np.c_[lg, pca.transform(Er)]
                    self.stack = (pca, LogisticRegression(C=.3, max_iter=500).fit(Zs, yr))
        return self
    def score(self, Tt, f, emb=None, ids=None):
        Z = self.feats(Tt, f); ids = Z.index if ids is None else pd.Index(ids, name="patient_id")
        ok = ids.isin(Z.index); p = pd.Series(self.rate, index=ids, dtype=float); src = pd.Series("train base rate (no current biologic)", index=ids)
        if ok.any():
            p[ok] = self.base.predict_proba(Z.reindex(ids[ok]))[:, 1]; src[ok] = "clinical"
            r = self._latest_rna(emb) if self.use_rna else None
            if r is not None:
                m = r.index.intersection(ids[ok])
                if len(m):
                    pca, lr = self.stack; Zs = np.c_[self._logit(p.loc[m].values), pca.transform(r.loc[m, self.rdims].values.astype(float))]
                    p.loc[m] = lr.predict_proba(Zs)[:, 1]; src.loc[m] = "clinical + rnaseq"
        out = pd.DataFrame({"patient_id": ids, "super_responder_prob": p.values, "sr_source": src.values}); assert out.super_responder_prob.between(0, 1).all()
        return out

In [50]:
# super responser detector

SR_MODEL = SRDetector(nb_mean, pre_c + early_c, sorted(cur_lines(Tc).drug_name.dropna().unique())).fit(Tc, f_all, y, emb=EMB); e_ = SR_MODEL.ev; EV["sr_model"] = e_

print(f"{e_['n']} biologic train patients | proxy super-responders {e_['pos']} ({e_['rate']:.1%}) | out-of-fold AUC {e_['auc']:.3f} | label-shuffle null AUC {e_['auc_null']:.3f} | Brier {e_['brier']:.4f} vs constant {e_['brier_const']:.4f} | mean prob {e_['mean_oof']:.3f}")
print("RNAseq stack:", SR_MODEL.rna_ev, "-> used" if SR_MODEL.use_rna else "-> NOT used (CI includes 0)")

check("V15 SR detector", "out-of-fold AUC >= 0.80 (7c level 2 reference 0.88)", e_["auc"] >= 0.80, f"{e_['auc']:.3f}")
check("V15 SR detector", "label-shuffle null AUC is near 0.5", abs(e_["auc_null"] - 0.5) < 0.05, f"{e_['auc_null']:.3f}")
check("V15 SR detector", "calibrated: mean out-of-fold probability within 0.02 of the observed rate", abs(e_["mean_oof"] - e_["rate"]) < 0.02, f"{e_['mean_oof']:.3f} vs {e_['rate']:.3f}")
check("V15 SR detector", "Brier score beats the constant-rate predictor", e_["brier"] < e_["brier_const"])

8548 biologic train patients | proxy super-responders 1008 (11.8%) | out-of-fold AUC 0.883 | label-shuffle null AUC 0.491 | Brier 0.0721 vs constant 0.1040 | mean prob 0.111
RNAseq stack: {'n': 600, 'pos': 49, 'auc_base': np.float64(0.7922700840771881), 'auc_stack': np.float64(0.936701359309604), 'lo': 0.09446343367299959, 'hi': 0.19694368571031415} -> used
PASS | V15 SR detector | out-of-fold AUC >= 0.80 (7c level 2 reference 0.88) | 0.883
PASS | V15 SR detector | label-shuffle null AUC is near 0.5 | 0.491
PASS | V15 SR detector | calibrated: mean out-of-fold probability within 0.02 of the observed rate | 0.111 vs 0.118
PASS | V15 SR detector | Brier score beats the constant-rate predictor | 


Patient report
Everything below is fitted on **train only** and works **per patient** (a patient's row never depends on the other test patients). Each part has its own validation in the ledger (`V16`) and its own evidence label.

| column(s) | how it is made | label |
|---|---|---|
| `p_worsen`, `worsen_flag` | calibrated gradient boosting on the forecast features + the current level + the forecast. **Worsening** = next-6-month IHS4 is at least 2 points **and** 25% above the patient's *current underlying level* (the shrunk non-flare level, baseline B7). Flag = probability above the threshold that maximises F1 out-of-fold. | CLINICAL (the signal is the forecast + level) |
| `sex`, `age_years`, `hurley_stage` | demographics table, age at 31 Dec 2025 (year of birth only). | - |
| `current_treatment`, `current_biologic`, `biologic_line`, `days_on_biologic`, `last_antibiotic` | treatments table at 31 Dec 2025: the open biologic line, the last antibiotic course, or the last stopped biologic. | - |
| `last_ihs4_score`, `last_score_date`, `last_score_age_days`, `last_score_visit_type` | last *scored* visit up to 31 Dec 2025. A score at a flare-reactive visit overstates the usual level (finding F4b). | ARTEFACT warning attached |
| `response_type`, `p_deep/partial/minimal`, `response_type_source` | for patients on a biologic: **observed** (Gaussian mixture on the shrunk on-drug / pre-drug ratio) when first-line scores exist before and after the start; otherwise **predicted** from the pre-treatment profile (weak: it only beats the prior, so most predicted calls are 'minimal' with low confidence). | CLINICAL when observed, UNDETERMINED when predicted |
| `top_risk_factor`, `top_clinical_factor`, `response_risk_factor` | **occlusion**: replace one group of features by the train median and see how many IHS4 points the forecast drops. The group with the largest drop is the biggest contributor. `top_clinical_factor` leaves out 'recent scores' and 'visit pattern' (which are measurements, not causes). `response_risk_factor` does the same on the response classifier (P minimal). | descriptive (what the model uses), not causal |
| `p_improve_if_surgery`, `p_improve_if_no_surgery`, `p_surgery_benefit` | Hurley 2-3 only. Two calibrated classifiers fitted on 120-day windows: *improvement = later non-flare level at least 25% below the earlier one*, one fitted on windows with a surgery in the first 30 days, one on windows without. Benefit = the extra probability of improving with surgery. | **UNDETERMINED**: confounded by indication (placebo test fails), adjusted effect is *worse*, not better. Settle with: a randomised or waiting-list comparison, or a pre-operative flare log. |
| `rnaseq_future_value`, `images_future_value` | out-of-fold MAE gain of the embedding correction in the matching group (months on biologic), with a bootstrap CI. 'No evidence' outside the population the correction was learned on. | CLINICAL (RNAseq); images UNDETERMINED (scope: recent starters; settled by test-set image patients) |


In [51]:
# define risk
RISK_LABEL = {"hurley": "Hurley stage (severity class)", "biologic": "biologic treatment status / time on drug", "visit_scores": "recent IHS4 scores",
              "visit_behaviour": "visit pattern (frequency, gaps)", "antibiotics": "antibiotic courses", "surgeries": "surgery history",
              "hidden_flare": "likely unrecorded flares (antibiotics/surgery without a visit)", "practitioner": "practitioner",
              "age_timing": "age and disease timing (age at diagnosis, delay, years since diagnosis)", "sex": "sex", "bmi": "BMI", "smoking": "smoking status",
              "ethnicity": "ethnicity", "family": "family history of HS"}

In [52]:
# function helper -- analysis

# risk group
def _risk_group(c, fam):
    if c in ("age", "yrs_since_dx", "age_dx", "delay", "prevalent"): return "age_timing"
    if c == "sex": return "sex"
    if c == "bmi": return "bmi"
    if c == "smoking_status": return "smoking"
    if c == "ethnicity": return "ethnicity"
    if c == "family_history_hs": return "family"
    return fam[c]

PRE_LABEL = {"hurley": "Hurley stage", "sex": "sex", "bmi": "BMI", "smoking_status": "smoking", "ethnicity": "ethnicity", "family_history_hs": "family history", "prevalent": "diagnosed before the data window", "age_dx": "age at diagnosis", "delay": "diagnostic delay"}
def _pre_group(c):
    if c in PRE_LABEL: return PRE_LABEL[c]
    if c in ("line", "drug"): return "line number / drug"
    if c in ("start_year", "age_start", "dx_to_start_days"): return "age and timing of the biologic start (years from diagnosis to start)"
    if c.startswith("pre_abx"): return "antibiotic build-up before the biologic"
    if c.startswith("pre_sur"): return "surgery build-up before the biologic"
    if c.startswith("pre_s_") or c.startswith("pre_v_") or c == "pre_flare_share": return "IHS4 level and visit pattern before the biologic"
    if c.startswith("early"): return "first 6 months on drug"
    return c

# panel
def _panel(T, anchors, y_window=True):
    """Patient x anchor-date table: state at the anchor (pre window) and, for training, the later level (post window) + surgery indicators."""
    D = np.timedelta64(1, "D"); dem = T["dem"].set_index("patient_id"); an = np.array([np.datetime64(a) for a in anchors]); out = []
    v = T["vis"]; v = v[v.ihs4_score.notna() & (v.visit_type != "flare_reactive")]
    vg = {k: (g.visit_date.values.astype("datetime64[ns]"), g.ihs4_score.values.astype(float)) for k, g in v.groupby("patient_id")}
    sg = {k: np.sort(g.procedure_date.values.astype("datetime64[ns]")) for k, g in T["sur"].groupby("patient_id")}
    ab = T["trt"][T["trt"].treatment_class == "Antibiotic"]; ag = {k: np.sort(g.start_date.values.astype("datetime64[ns]")) for k, g in ab.groupby("patient_id")}
    bt = T["trt"][T["trt"].treatment_class == "Biologic"]; bg = {k: (g.start_date.values.astype("datetime64[ns]"), g.end_date.values.astype("datetime64[ns]")) for k, g in bt.groupby("patient_id")}
    empty = np.array([], dtype="datetime64[ns]")
    def cnt(arr, lo, hi): return np.searchsorted(arr, hi, "left") - np.searchsorted(arr, lo, "left")
    for p in dem.index:
        dt, sc = vg.get(p, (empty, np.array([]))); cs = np.r_[0, np.cumsum(sc)]
        def wmean(lo, hi):
            i, j = np.searchsorted(dt, lo, "left"), np.searchsorted(dt, hi, "left"); n = j - i
            return np.where(n > 0, (cs[j] - cs[i]) / np.maximum(n, 1), np.nan), n
        sx, ax = sg.get(p, empty), ag.get(p, empty); bs, be = bg.get(p, (empty, empty))
        pre_m, pre_n = wmean(an - 180 * D, an - 7 * D); pp_m, pp_n = wmean(an - 360 * D, an - 180 * D)
        r = dict(patient_id=np.full(len(an), p), anchor=an, pre_m=pre_m, pre_n=pre_n, prepre_m=pp_m, s_n_y=cnt(sx, an - 365 * D, an), s_near=cnt(sx, an - 120 * D, an), s_early=cnt(sx, an, an + 30 * D), s_late=cnt(sx, an + 30 * D, an + 240 * D), abx_180=cnt(ax, an - 180 * D, an))
        if y_window: r["post_m"], r["post_n"] = wmean(an + 60 * D, an + 240 * D)
        ob = np.zeros(len(an), bool); dob = np.full(len(an), np.nan)
        for s_, e_ in zip(bs, be):
            m = (an >= s_) & (np.isnat(e_) | (an < e_)); ob |= m; dob = np.where(m, (an - s_) / D, dob)
        r["onb"] = ob.astype(float); r["days_on_bio"] = dob; out.append(pd.DataFrame(r))
    P = pd.concat(out, ignore_index=True); d = dem.reindex(P.patient_id)
    P["hurley"] = d.hurley_stage_baseline.values; P["male"] = (d.sex.values == "Male").astype(float); P["age"] = pd.to_datetime(P.anchor).dt.year.values - d.birth_year.values; P["bmi"] = d.bmi_at_baseline.values
    return P
SURG_X = ["pre_m", "pre_n", "prepre_m", "s_n_y", "abx_180", "onb", "days_on_bio", "hurley", "male", "age", "bmi"]

class PatientReport:
    def __init__(self, model, sr): self.m, self.sr, self.ev, self.notes = model, sr, {}, {}
    # ---- current underlying level (the B7 baseline: decayed non-flare score level shrunk toward the Hurley x biologic median)
    def level0(self, f):
        w = f["s_nonflare_ewm_w_120"].fillna(0).values; e = f["s_nonflare_ewm_120"].fillna(0).values
        an = self.anchor.reindex(pd.MultiIndex.from_arrays([f.hurley.values, f.on_bio.values])).values; an = np.where(np.isnan(an), self.ymed, an)
        return pd.Series((w * e + self.k * an) / (w + self.k), index=f.index)
    def fit(self, X, y, oof_m0, Tc, f_all, EMB, RT, gm, order, seed=0):
        cols = self.m.cols; self.ymed = float(y.median()); self.k = float(np.median(EV["eb_ks"])); self.cols = cols
        self.anchor = y.groupby([X.hurley, X.on_bio]).median(); self.fam = families(cols)
        self.ref_med = X[cols].median()
        # ===== 1. worsening
        L0 = self.level0(X); self.delta = lambda l: np.maximum(2.0, 0.25 * l)
        lab = ((y - L0) >= self.delta(L0)).astype(int); self.ev["worsen_rate"] = float(lab.mean())
        W = X[cols].assign(level0=L0, base=pd.Series(np.asarray(oof_m0), index=X.index)); self.wcols = list(W.columns)
        mk = lambda: CalibratedClassifierCV(HGBC(max_iter=200, learning_rate=.05, max_leaf_nodes=8, min_samples_leaf=40, random_state=0), method="isotonic", cv=5)
        self.w = mk().fit(W, lab); oof = np.zeros(len(W))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(W, lab): oof[te] = mk().fit(W.iloc[tr], lab.iloc[tr]).predict_proba(W.iloc[te])[:, 1]
        best = max(np.linspace(.1, .7, 61), key=lambda t: 2 * ((oof >= t) & (lab == 1)).sum() / max(((oof >= t).sum() + (lab == 1).sum()), 1)); self.thr = float(best)
        fl = oof >= self.thr; self.ev["worsen"] = dict(auc=roc_auc_score(lab, oof), brier=float(np.mean((oof - lab) ** 2)), brier_const=float(lab.mean() * (1 - lab.mean())), thr=self.thr, flag_share=float(fl.mean()), precision=float(lab.values[fl].mean()) if fl.any() else np.nan, recall=float(fl[lab.values == 1].mean()), mean_oof=float(oof.mean()))
        self.ev["worsen"]["auc_forecast_only"] = roc_auc_score(lab, np.asarray(oof_m0) - L0.values)       # does the plain 'forecast minus level' rule do as well?
        # ===== 2. response type: observed when first-line scores exist, else predicted from pre-treatment information
        self.gm, self.order, self.names = gm, order, np.array(["deep", "partial", "minimal"])
        h = f_all.hurley.reindex(RT.index); self.eb = {}
        for nm in ("pre", "on"):
            self.eb[nm] = {}
            for g in h.dropna().unique():
                i = h == g; mm, nn, vv = RT.loc[i, f"{nm}_m"], RT.loc[i, f"{nm}_n"], RT.loc[i, f"{nm}_v"]
                s2w = np.nanmean(vv[nn >= 2]); s2b = max(mm.var() - np.nanmean(s2w / nn), 1e-3); self.eb[nm][g] = (mm.mean(), s2w / s2b)
        Z = self.sr.feats(Tc, f_all); self.pre_cols = [c for c in Z.columns if not c.startswith("early")]
        Zr = Z.reindex(RT.index)[self.pre_cols]; yt = pd.Categorical(RT.type, categories=list(self.names)).codes
        mkc = lambda: HGBC(max_iter=150, learning_rate=.05, max_leaf_nodes=6, min_samples_leaf=40, random_state=0)
        self.rc = mkc().fit(Zr, yt); o = np.zeros((len(Zr), 3))
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Zr, yt): o[te] = mkc().fit(Zr.iloc[tr], yt[tr]).predict_proba(Zr.iloc[te])
        prior = np.bincount(yt) / len(yt); ll = lambda P_: -np.mean(np.log(np.clip(P_[np.arange(len(yt)), yt], 1e-6, 1)))
        self.ev["resp"] = dict(n=len(yt), shares=dict(zip(self.names, prior.round(3))), acc=float((o.argmax(1) == yt).mean()), acc_majority=float(prior.max()), logloss=ll(o), logloss_prior=ll(np.tile(prior, (len(yt), 1))),
                               auc_deep=roc_auc_score(yt == 0, o[:, 0]), auc_minimal=roc_auc_score(yt == 2, o[:, 2]))
        self.pre_ref = Zr.median()
        # ===== 3. surgery: descriptive, adjusted and placebo-tested
        self._fit_surgery(Tc, seed)
        # ===== 4. what would RNAseq / images add in the future (from the train OOF gains)
        self._fit_embvalue(X, y, oof_m0, EMB, seed)
        return self

    # -- surgery
    def _fit_surgery(self, Tc, seed):
        anchors = pd.date_range("2018-03-01", "2025-03-31", freq="120D"); P = _panel(Tc, anchors)
        P = P[(P.pre_n >= 1) & (P.post_n >= 1) & (P.pre_m >= 3) & P.hurley.isin([2, 3])].copy()
        tr_ = (P.s_early > 0) & (P.s_near == 0); co_ = (P.s_early == 0) & (P.s_near == 0) & (P.s_late == 0); P = P[tr_ | co_].copy(); P["sur"] = tr_[P.index].astype(int)
        P["improved"] = (P.post_m <= 0.75 * P.pre_m).astype(int); P["gain"] = P.post_m - P.pre_m
        grp = P.patient_id.values; self.surg_n = (int(P.sur.sum()), int((1 - P.sur).sum()))
        mk = lambda: CalibratedClassifierCV(HGBC(max_iter=150, learning_rate=.05, max_leaf_nodes=6, min_samples_leaf=40, random_state=0), method="isotonic", cv=4)
        T1, C0 = P[P.sur == 1], P[P.sur == 0]
        self.s1 = mk().fit(T1[SURG_X], T1.improved); self.s0 = mk().fit(C0[SURG_X], C0.improved)
        # out-of-fold evaluation by patient groups
        o1, o0 = np.zeros(len(P)), np.zeros(len(P))
        for tr, te in GroupKFold(4).split(P, groups=grp):
            a, b = P.iloc[tr], P.iloc[te]; a1, a0 = a[a.sur == 1], a[a.sur == 0]
            o1[te] = mk().fit(a1[SURG_X], a1.improved).predict_proba(b[SURG_X])[:, 1]; o0[te] = mk().fit(a0[SURG_X], a0.improved).predict_proba(b[SURG_X])[:, 1]
        t_ = (P.sur == 1).values; up = o1 - o0
        # propensity (overlap): can the covariates tell surgery cases from controls?
        pr = np.zeros(len(P))
        for tr, te in GroupKFold(4).split(P, groups=grp): pr[te] = HGBC(max_iter=100, learning_rate=.05, max_leaf_nodes=6, random_state=0).fit(P.iloc[tr][SURG_X], P.sur.iloc[tr]).predict_proba(P.iloc[te][SURG_X])[:, 1]
        # adjusted shift of the later level and a placebo (effect of a FUTURE surgery on the level BEFORE the anchor = pure confounding)
        def ols(Q, yc):
            Q = Q.dropna(subset=[yc]); Xm = np.c_[np.ones(len(Q)), Q.pre_m, Q.sur, Q.hurley == 3, Q.onb, Q.s_n_y, Q.abx_180]; return np.linalg.lstsq(Xm, Q[yc].values, rcond=None)[0][2]
        rng = np.random.default_rng(seed); ids = P.patient_id.unique(); G = {k: g for k, g in P.groupby("patient_id")}; bs_post, bs_pl = [], []
        for _ in range(40):
            Q = pd.concat([G[k] for k in rng.choice(ids, len(ids))]); bs_post.append(ols(Q, "post_m")); bs_pl.append(ols(Q.assign(pre_m=Q.pre_m), "prepre_m"))
        ci = lambda b: (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))
        self.ev["surg"] = dict(n_surgery=self.surg_n[0], n_control=self.surg_n[1], rate_improved_surgery=float(T1.improved.mean()), rate_improved_control=float(C0.improved.mean()),
                               auc_p1=roc_auc_score(P.improved[t_], o1[t_]), auc_p0=roc_auc_score(P.improved[~t_], o0[~t_]), mean_uplift_treated=float(up[t_].mean()), mean_uplift_control=float(up[~t_].mean()),
                               propensity_auc=roc_auc_score(P.sur, pr), ancova_shift=float(ols(P, "post_m")), ancova_ci=ci(bs_post), placebo_shift=float(ols(P, "prepre_m")), placebo_ci=ci(bs_pl))
        e = self.ev["surg"]; self.surg_confounded = not (e["placebo_ci"][0] <= 0 <= e["placebo_ci"][1]); self.surg_no_benefit = e["ancova_ci"][0] > -0.2

    # -- embeddings: expected value of doing the test
    def _fit_embvalue(self, X, y, oof_m0, EMB, seed):
        res = pd.Series(y.values - np.asarray(oof_m0), index=X.index); self.embv = {}; self.rna_days, self.img_days = 365, 180
        bins = [-1, 90, 180, 365, 99999]; labs = ["0-90 d on biologic", "90-180 d", "180-365 d", ">365 d"]
        for key in ("rnaseq", "images"):
            m = EMB[key]; m = m[m.patient_id.isin(X.index)]; dc = "collection_date" if key == "rnaseq" else "capture_date"; m = m.sort_values(dc).drop_duplicates("patient_id", keep="last")
            dims = [c for c in m if "_dim_" in c]; E = m[dims].values.astype(float); r = res.reindex(m.patient_id).values; xx = X.reindex(m.patient_id)
            seg = pd.cut(xx.days_on_bio.where(xx.on_bio == 1).values, bins, labels=labs).astype(object); seg = np.where(xx.on_bio.values == 1, seg, "not on biologic")
            insg = (xx.on_bio.values == 1) & (xx.days_on_bio.values < (self.img_days if key == "images" else 10 ** 6))
            o = np.zeros(len(E)); o_seg = np.zeros(len(E))
            for tr, te in KFold(5, shuffle=True, random_state=seed).split(E):
                o[te] = mk_corr(16).fit(E[tr], r[tr]).predict(E[te])
                if key == "images": trm = tr[insg[tr]]; o_seg[te] = mk_corr(16).fit(E[trm], r[trm]).predict(E[te])
            rng = np.random.default_rng(seed); rows = {}
            for s_ in pd.unique(seg):
                use = o_seg if (key == "images" and s_ in ("0-90 d on biologic", "90-180 d")) else o        # images: judged with the segment-limited corrector that is actually applied
                g = (np.abs(r) - np.abs(r - use))[seg == s_]; bb = rng.choice(g, (500, len(g))).mean(1) if len(g) > 5 else np.array([np.nan, np.nan])
                rows[s_] = dict(n=int(len(g)), gain=float(g.mean()), lo=float(np.nanpercentile(bb, 2.5)), hi=float(np.nanpercentile(bb, 97.5)))
            self.embv[key] = rows; self.ev[f"{key}_value"] = rows
            if key == "images":
                trm = np.where(insg)[0]; self.img_corr = (dims, mk_corr(16).fit(E[trm], r[trm]))
                # robustness of the segment-limited image correction: leave-device-out and a shuffled-embedding null (both inside the segment)
                dev_ = m.device_id.values; o_d = np.zeros(len(E)); o_n = np.zeros(len(E)); En = E[np.random.default_rng(seed + 1).permutation(len(E))]
                for tr, te in GroupKFold(5).split(E, groups=dev_):
                    trm_ = tr[insg[tr]]; o_d[te] = mk_corr(16).fit(E[trm_], r[trm_]).predict(E[te])
                for tr, te in KFold(5, shuffle=True, random_state=seed).split(E): trm_ = tr[insg[tr]]; o_n[te] = mk_corr(16).fit(En[trm_], r[trm_]).predict(En[te])
                gg = lambda oo: (lambda g: dict(gain=float(g.mean()), lo=float(np.percentile(rng.choice(g, (500, len(g))).mean(1), 2.5)), hi=float(np.percentile(rng.choice(g, (500, len(g))).mean(1), 97.5)), n=int(len(g))))((np.abs(r) - np.abs(r - oo))[insg])
                self.ev["img_seg"] = dict(plain=gg(o_seg), leave_device_out=gg(o_d), shuffled_null=gg(o_n))

    # -- resolve per patient
    def _obs_response(self, Tc, hurley):
        cur = cur_lines(Tc); v = Tc["vis"][Tc["vis"].ihs4_score.notna()].join(cur.start_date.rename("t0"), on="patient_id", how="inner"); d = (v.visit_date - v.t0).dt.days
        parts = {}
        for nm, x in [("pre", v[(d < 0) & (d >= -365)]), ("on", v[d >= 90])]:
            g = x.groupby("patient_id").ihs4_score; parts[nm] = pd.DataFrame({f"{nm}_n": g.size(), f"{nm}_m": g.mean()})
        r = pd.concat(parts.values(), axis=1, join="inner"); h = hurley.reindex(r.index); out = pd.DataFrame(index=r.index)
        for nm in ("pre", "on"):
            mu = h.map({g: p[0] for g, p in self.eb[nm].items()}); kk = h.map({g: p[1] for g, p in self.eb[nm].items()}); out[f"{nm}_s"] = (r[f"{nm}_n"] * r[f"{nm}_m"] + kk * mu) / (r[f"{nm}_n"] + kk)
        out["lr"] = np.log(out.on_s / out.pre_s); out["line"] = cur.treatment_line.reindex(out.index); out["n_pre"] = r.pre_n; out["n_on"] = r.on_n
        return out.replace([np.inf, -np.inf], np.nan).dropna(subset=["lr"])
    def make(self, Tt, f, ids, scen, emb_t=None, sr_prob=None):
        """f: features built on `ids` with build(); scen: dict of Series from predict_scenarios (indexed by patient_id)."""
        ids = pd.Index(ids, name="patient_id"); f = f.reindex(ids); out = pd.DataFrame(index=ids); dem = Tt["dem"].set_index("patient_id").reindex(ids)
        # identity
        out["sex"] = dem.sex.values; out["age_years"] = (END.year - dem.birth_year).values; out["hurley_stage"] = f.hurley.values
        out["years_since_diagnosis"] = f.yrs_since_dx.values
        # current treatment
        trt = Tt["trt"]; cur = cur_lines(Tt).reindex(ids); out["on_biologic"] = ids.isin(cur_lines(Tt).index).astype(int)
        out["current_biologic"] = np.where(out.on_biologic == 1, cur.drug_name.values, "none"); out["biologic_line"] = cur.treatment_line.values
        out["days_on_biologic"] = (END - cur.start_date).dt.days.values
        ab = trt[trt.treatment_class == "Antibiotic"].sort_values("start_date").groupby("patient_id").tail(1).set_index("patient_id").reindex(ids)
        out["last_antibiotic"] = ab.drug_name.values; out["days_since_antibiotic_end"] = (END - ab.end_date.fillna(ab.start_date)).dt.days.values
        out["antibiotic_active_now"] = (((END - ab.end_date.fillna(END)).dt.days <= 14).values & ab.drug_name.notna().values).astype(int)
        stopped = trt[(trt.treatment_class == "Biologic") & trt.end_date.notna()].sort_values("end_date").groupby("patient_id").tail(1).set_index("patient_id").reindex(ids)
        out["last_stopped_biologic"] = np.where(out.on_biologic == 0, stopped.drug_name.values, None); out["days_since_biologic_stop"] = np.where(out.on_biologic == 0, (END - stopped.end_date).dt.days.values, np.nan)
        def txt(i):
            r = out.iloc[i]
            s = f"{r.current_biologic} (line {int(r.biologic_line)}, {int(r.days_on_biologic)} d on drug)" if r.on_biologic else ("no biologic" + (f"; last: {r.last_stopped_biologic} stopped {int(r.days_since_biologic_stop)} d ago" if pd.notna(r.last_stopped_biologic) else "; never on biologic"))
            if r.antibiotic_active_now: s += f"; antibiotic: {r.last_antibiotic} (active)"
            elif pd.notna(r.last_antibiotic) and r.days_since_antibiotic_end <= 90: s += f"; antibiotic: {r.last_antibiotic} ended {int(r.days_since_antibiotic_end)} d ago"
            return s
        out["current_treatment"] = [txt(i) for i in range(len(out))]
        # last score
        v = Tt["vis"][Tt["vis"].ihs4_score.notna() & (Tt["vis"].visit_date <= END)].sort_values("visit_date").groupby("patient_id").tail(1).set_index("patient_id").reindex(ids)
        out["last_ihs4_score"] = v.ihs4_score.values; out["last_score_date"] = v.visit_date.dt.date.values; out["last_score_age_days"] = (END - v.visit_date).dt.days.values; out["last_score_visit_type"] = v.visit_type.values
        out["scored_visits_365d"] = f.v_scored_365.values
        # forecast + level + worsening
        out["pred_avg_ihs4_next6"] = scen["final"].reindex(ids).values; L0 = self.level0(f); out["current_level_ihs4"] = L0.values
        out["expected_change"] = out.pred_avg_ihs4_next6 - out.current_level_ihs4
        W = f[self.cols].assign(level0=L0, base=scen["M0"].reindex(ids).values)[self.wcols]; pw = self.w.predict_proba(W)[:, 1]
        out["p_worsen"] = pw; out["worsen_flag"] = (pw >= self.thr).astype(int)
        # risk factors by occlusion of the forecast
        base = predict_blend(self.m.models, f, self.cols); contrib = {}
        gmap = {c: _risk_group(c, self.fam) for c in self.cols}
        for g in sorted(set(gmap.values())):
            cc = [c for c in self.cols if gmap[c] == g]; f2 = f[self.cols].copy(); f2[cc] = self.ref_med[cc].values; contrib[g] = base - predict_blend(self.m.models, f2, self.cols)
        C = pd.DataFrame(contrib, index=ids); top = C.values.argsort(1)[:, ::-1]; gl = np.array(C.columns)
        out["top_risk_factor"] = [RISK_LABEL.get(gl[t[0]], gl[t[0]]) for t in top]; out["top_risk_points"] = C.values[np.arange(len(C)), top[:, 0]]
        out["second_risk_factor"] = [RISK_LABEL.get(gl[t[1]], gl[t[1]]) for t in top]; out["second_risk_points"] = C.values[np.arange(len(C)), top[:, 1]]
        out.loc[out.top_risk_points <= 0.05, ["top_risk_factor"]] = "none stands out"
        # the same, but leaving out the two 'measurement' groups (recent scores, visit pattern): what is it about the patient's disease and care?
        keep = [i for i, g in enumerate(gl) if g not in ("visit_scores", "visit_behaviour")]; Ck = C.values[:, keep]; bi = Ck.argmax(1)
        out["top_clinical_factor"] = [RISK_LABEL.get(gl[keep[b]], gl[keep[b]]) if Ck[j, b] > 0.05 else "none stands out" for j, b in enumerate(bi)]; out["top_clinical_points"] = Ck[np.arange(len(Ck)), bi]
        # super-responder probability (biologic patients only)
        out["super_responder_prob"] = np.nan
        if sr_prob is not None: out["super_responder_prob"] = sr_prob.reindex(ids).values
        # response type (biologic patients)
        out["response_type"] = None; out["response_type_source"] = None; out["response_confidence"] = np.nan; out["p_deep"] = np.nan; out["p_partial"] = np.nan; out["p_minimal"] = np.nan; out["response_risk_factor"] = None
        bio = ids[out.on_biologic.values == 1]
        if len(bio):
            ob = self._obs_response(Tt, f.hurley); ob = ob[(ob.line == 1) & ob.index.isin(bio)]
            Zp = self.sr.feats(Tt, f).reindex(bio)[self.pre_cols]; pp = pd.DataFrame(self.rc.predict_proba(Zp), index=bio, columns=self.names)
            if len(ob):
                po = self.gm.predict_proba(ob[["lr"]])[:, self.order]; po = pd.DataFrame(po, index=ob.index, columns=self.names)
            for p_ in bio:
                if len(ob) and p_ in ob.index: pr, src = po.loc[p_], f"observed (first line; {int(ob.loc[p_, 'n_pre'])} scores before, {int(ob.loc[p_, 'n_on'])} on drug)"
                else: pr, src = pp.loc[p_], "predicted from pre-treatment profile" + ("" if out.loc[p_, "biologic_line"] == 1 else " (model learned on first-line patients)")
                out.loc[p_, ["p_deep", "p_partial", "p_minimal"]] = pr.values; out.loc[p_, "response_type"] = self.names[int(np.argmax(pr.values))]; out.loc[p_, "response_type_source"] = src; out.loc[p_, "response_confidence"] = float(np.max(pr.values))
            # biggest pre-treatment factor pushing toward minimal response (occlusion of the response classifier)
            pm = self.rc.predict_proba(Zp)[:, 2]; cg = {c: _pre_group(c) for c in self.pre_cols}; cont = {}
            for g in sorted(set(cg.values())):
                cc = [c for c in self.pre_cols if cg[c] == g]; z2 = Zp.copy(); z2[cc] = self.pre_ref[cc].values; cont[g] = pm - self.rc.predict_proba(z2)[:, 2]
            Cr = pd.DataFrame(cont, index=bio); out.loc[bio, "response_risk_factor"] = [Cr.columns[i] if Cr.values[j, i] > 0.01 else "none stands out" for j, i in enumerate(Cr.values.argmax(1))]
        # surgery (Hurley 2 and 3)
        for c in ("p_improve_if_surgery", "p_improve_if_no_surgery", "p_surgery_benefit"): out[c] = np.nan
        out["surgery_note"] = None; h23 = (out.hurley_stage.isin([2, 3])).values
        if h23.any():
            P = _panel(Tt, [END], y_window=False).set_index("patient_id").reindex(ids[h23]); P["sur"] = 0.0
            q1, q0 = self.s1.predict_proba(P[SURG_X])[:, 1], self.s0.predict_proba(P[SURG_X])[:, 1]; ok = P.pre_m.notna().values & (P.pre_m.fillna(0) >= 3).values
            ii = np.where(h23)[0]
            for k_, j in enumerate(ii):
                if ok[k_]: out.iloc[j, out.columns.get_loc("p_improve_if_surgery")] = q1[k_]; out.iloc[j, out.columns.get_loc("p_improve_if_no_surgery")] = q0[k_]; out.iloc[j, out.columns.get_loc("p_surgery_benefit")] = max(q1[k_] - q0[k_], 0)
                else: out.iloc[j, out.columns.get_loc("surgery_note")] = "no recent non-flare score >= 3 to base this on"
            cav = "association only (confounded by indication): do not read as a causal benefit" if self.surg_confounded else "association, adjusted for level; not randomised"
            out.loc[out.surgery_note.isna() & h23, "surgery_note"] = cav
        # imaging / RNAseq future value
        has_r = ids.isin(set((emb_t or {}).get("rnaseq", pd.DataFrame({"patient_id": []})).patient_id)); has_i = ids.isin(set((emb_t or {}).get("images", pd.DataFrame({"patient_id": []})).patient_id))
        out["has_rnaseq"] = has_r.astype(int); out["has_images"] = has_i.astype(int)
        def seg_of(i):
            if out.on_biologic.iloc[i] != 1: return "not on biologic"
            d = out.days_on_biologic.iloc[i]; return "0-90 d on biologic" if d < 90 else "90-180 d" if d < 180 else "180-365 d" if d < 365 else ">365 d"
        for key, nm, flag in [("rnaseq", "RNAseq", has_r), ("images", "imaging", has_i)]:
            val, gn = [], []
            for i in range(len(out)):
                s = seg_of(i); r = self.embv[key].get(s)
                if flag[i]: val.append("already available"); gn.append(np.nan)
                elif r is None or r["n"] < 30 or (key == "rnaseq" and (s == "not on biologic" or out.days_on_biologic.iloc[i] > self.rna_days)): val.append("no evidence (outside the trained population)"); gn.append(np.nan)
                elif r["lo"] > 0: val.append(f"likely to improve the forecast (+{r['gain']:.2f} IHS4 points MAE; 95% CI {r['lo']:.2f}..{r['hi']:.2f})"); gn.append(r["gain"])
                elif r["hi"] < 0: val.append(f"likely to worsen the forecast ({r['gain']:.2f}; CI {r['lo']:.2f}..{r['hi']:.2f}): do not add"); gn.append(r["gain"])
                else: val.append(f"no detectable gain (CI {r['lo']:.2f}..{r['hi']:.2f})"); gn.append(r["gain"])
            out[f"{key}_future_value"] = val; out[f"{key}_expected_mae_gain"] = gn
        e = out.on_biologic.eq(1) & out.days_on_biologic.le(365) & ~has_r
        out["rnaseq_for_response_prediction"] = np.where(e, f"may help spot super-responders early (train RNAseq cohort AUC {self.sr.rna_ev['auc_base']:.2f} -> {self.sr.rna_ev['auc_stack']:.2f}; small sample)" if self.sr.use_rna else "not supported", "")
        return out.reset_index()

# scenario predictor
def predict_scenarios(model, Tt, ids, emb_t=None, rep=None, f=None):
    # Forecast per patient under four information sets: M0 (tables only), M0+images, M0+rnaseq, final (RNAseq if present, else images).
    # Embedding corrections are applied ONLY inside the population they were learned on (on a biologic; RNAseq <= rep.rna_days, images < rep.img_days since start).
    ids = pd.Index(ids, name="patient_id"); f = build(Tt, END, uplift=model.uplift, ids=ids) if f is None else f.reindex(ids)
    base = pd.Series(predict_blend(model.models, f, model.cols), index=ids); emb_t = emb_t or {}
    rd, idl = (rep.rna_days, rep.img_days) if rep is not None else (10 ** 6, 10 ** 6)
    seg_r = ((f.on_bio == 1) & (f.days_on_bio <= rd)).values; seg_i = ((f.on_bio == 1) & (f.days_on_bio < idl)).values
    def apply(key, dims, cm, seg):
        pred = base.copy(); used = []
        if key in emb_t:
            dc = "collection_date" if key == "rnaseq" else "capture_date"; m = emb_t[key].sort_values(dc).drop_duplicates("patient_id", keep="last"); m = m[m.patient_id.isin(ids[seg])]
            if len(m): pred.loc[m.patient_id.values] += cm.predict(m[dims].values.astype(float)); used = list(m.patient_id)
        return pred, used
    pr, ur = (apply("rnaseq", *model.corr["rnaseq"], seg_r) if "rnaseq" in model.corr else (base.copy(), []))
    pi, ui = (apply("images", *rep.img_corr, seg_i) if (rep is not None and getattr(rep, "img_corr", None)) else (base.copy(), []))
    fin = pr.copy(); rest = [p for p in ui if p not in set(ur)]
    if rest: fin.loc[rest] = pi.loc[rest]
    out = {"M0": base, "M0+rnaseq": pr, "M0+images": pi, "final": fin}; out = {k: v.clip(lower=0) for k, v in out.items()}
    out["f"] = f; out["rna_used"] = ur; out["img_used"] = ui
    return out


In [53]:
# create the report
REPORT = PatientReport(FINAL, SR_MODEL).fit(X, y, OOF["M0 blend"], Tc, f_all, EMB, RT, gm, order)
e = REPORT.ev; w = e["worsen"]; r_ = e["resp"]; s_ = e["surg"]; im = e["img_seg"]

print(f"WORSENING   label rate {e['worsen_rate']:.1%} | OOF AUC {w['auc']:.3f} (plain 'forecast minus level' rule {w['auc_forecast_only']:.3f}) | flag at p >= {w['thr']:.2f}: {w['flag_share']:.1%} flagged, precision {w['precision']:.2f}, recall {w['recall']:.2f}")

print(f"RESPONSE    {r_['n']} typed first-line patients, shares {r_['shares']} | pre-treatment classifier: accuracy {r_['acc']:.3f} (majority {r_['acc_majority']:.3f}) | log-loss {r_['logloss']:.3f} vs prior {r_['logloss_prior']:.3f} | AUC deep {r_['auc_deep']:.2f}, minimal {r_['auc_minimal']:.2f}")

print(f"SURGERY     {s_['n_surgery']} surgeries vs {s_['n_control']} control windows (Hurley 2-3) | improved >=25%: {s_['rate_improved_surgery']:.1%} with surgery vs {s_['rate_improved_control']:.1%} without | adjusted shift of later level {s_['ancova_shift']:+.2f} (CI {s_['ancova_ci'][0]:+.2f}..{s_['ancova_ci'][1]:+.2f}) | placebo shift {s_['placebo_shift']:+.2f} (CI {s_['placebo_ci'][0]:+.2f}..{s_['placebo_ci'][1]:+.2f}) | overlap AUC {s_['propensity_auc']:.2f}")

print("RNAseq value by months on biologic:", {k: f"{v['gain']:+.2f} (n={v['n']})" for k, v in e["rnaseq_value"].items()})

print("IMAGE value (segment-limited corrector):", {k: f"{v['gain']:+.2f} (n={v['n']})" for k, v in e["images_value"].items()}); print("  images, <180 d on biologic:", {k: f"{v['gain']:+.3f} ({v['lo']:+.3f}..{v['hi']:+.3f})" for k, v in im.items()})

check("V16 patient report", "worsening classifier: out-of-fold AUC >= 0.70 and better than the plain forecast-minus-level rule", w["auc"] >= 0.70 and w["auc"] > w["auc_forecast_only"], f"{w['auc']:.3f} vs {w['auc_forecast_only']:.3f}")

check("V16 patient report", "worsening classifier is calibrated (mean probability within 0.02 of the observed rate) and beats the constant Brier score", abs(w["mean_oof"] - e["worsen_rate"]) < 0.02 and w["brier"] < w["brier_const"], f"mean {w['mean_oof']:.3f} vs {e['worsen_rate']:.3f}; Brier {w['brier']:.4f} vs {w['brier_const']:.4f}")

check("V16 patient report", "response-type classifier beats the prior (log-loss) - used only when first-line scores are not available", r_["logloss"] < r_["logloss_prior"], f"{r_['logloss']:.3f} vs {r_['logloss_prior']:.3f}; accuracy is NOT better than always saying 'minimal' ({r_['acc']:.3f} vs {r_['acc_majority']:.3f})")

check("V16 patient report", "surgery: no confounding detected by the placebo test (a FUTURE surgery must not change the PAST level)", not REPORT.surg_confounded, f"placebo shift {s_['placebo_shift']:+.2f} (CI {s_['placebo_ci'][0]:+.2f}..{s_['placebo_ci'][1]:+.2f}) -> association only", warn=True)

check("V16 patient report", "surgery: surgery cases and controls overlap (propensity AUC < 0.85)", s_["propensity_auc"] < 0.85, f"AUC {s_['propensity_auc']:.2f}", warn=True)

check("V16 patient report", "images: gain inside the validated segment (<180 d on a biologic) has CI above 0", im["plain"]["lo"] > 0, f"{im['plain']['gain']:+.3f} ({im['plain']['lo']:+.3f}..{im['plain']['hi']:+.3f}), n={im['plain']['n']}")

check("V16 patient report", "images: the gain survives leave-device-out and the shuffled-embedding null is ~0", im["leave_device_out"]["lo"] > 0 and abs(im["shuffled_null"]["gain"]) < 0.05, f"leave-device-out {im['leave_device_out']['gain']:+.3f}; null {im['shuffled_null']['gain']:+.3f}")

check("V16 patient report", "RNAseq: gain in the first 180 days on a biologic has CI above 0", all(e["rnaseq_value"][k]["lo"] > 0 for k in ("0-90 d on biologic", "90-180 d")), "")

WORSENING   label rate 10.1% | OOF AUC 0.775 (plain 'forecast minus level' rule 0.747) | flag at p >= 0.18: 14.4% flagged, precision 0.30, recall 0.43
RESPONSE    4741 typed first-line patients, shares {np.str_('deep'): np.float64(0.146), np.str_('partial'): np.float64(0.232), np.str_('minimal'): np.float64(0.622)} | pre-treatment classifier: accuracy 0.614 (majority 0.622) | log-loss 0.874 vs prior 0.916 | AUC deep 0.71, minimal 0.67
SURGERY     3562 surgeries vs 41534 control windows (Hurley 2-3) | improved >=25%: 14.7% with surgery vs 27.7% without | adjusted shift of later level +2.25 (CI +2.16..+2.34) | placebo shift +0.15 (CI +0.04..+0.28) | overlap AUC 0.80
RNAseq value by months on biologic: {'>365 d': '-0.21 (n=5)', '180-365 d': '+0.03 (n=98)', '90-180 d': '+0.35 (n=219)', '0-90 d on biologic': '+0.50 (n=278)'}
IMAGE value (segment-limited corrector): {'not on biologic': '-0.03 (n=840)', '>365 d': '-0.03 (n=1184)', '180-365 d': '+0.00 (n=253)', '90-180 d': '+0.12 (n=244)', '0-

## 10. test set input and output path

Predicts **only those patient ids** mentioned in the file, fills each template (rows, order, columns unchanged), and writes one extra file `patient_report_<TEST_NAME>.csv` with the clinical columns of section 10e. All checks go to the ledger (steps starting `H handler`).

* **Exercise 1 and 2** = the forecast **without** embeddings (`ex1_uses_emb=False`), so that Exercise 3 can be compared with them on the same patients. 
* **Exercise 3a (images)** = forecast + image correction, applied only to patients on a biologic for < 180 days (the only group where it helped: +0.21 MAE, survives leave-device-out). 
* **Exercise 3b (images + RNAseq)** = forecast + RNAseq correction for patients on a biologic for <= 365 days. The image correction is **not** added on top: where both exist, images add nothing beyond RNAseq (they make it worse by 0.16).

Else specified, everybody else keeps the Exercise 1 value.

In [54]:
# functions helper -- general

def url_exists(d, name):
    p = _src(d, name)
    if not str(p).startswith("http"): return os.path.exists(p)
    try:
        with urllib.request.urlopen(urllib.request.Request(p, method="HEAD")): return True
    except (urllib.error.HTTPError, urllib.error.URLError): return False

def read_template(path, cols):
    t = pd.read_csv(str(path), dtype={"patient_id": str}); return t

def read_template(path, cols):
    path = str(path)
    if not path.startswith("http") and not os.path.exists(path):
        raise FileNotFoundError(f"Template not found: {path}")
    t = pd.read_csv(path, dtype={"patient_id": str})
    miss = [c for c in cols if c not in t.columns]
    assert not miss, f"template lacks columns {miss}; has {list(t.columns)}"
    return t

In [55]:
# function helper -- analysis

def run_template_handler(test_dir, ex1, ex2=None, ex3a=None, ex3b=None, model=None, sr=None, rep=None, out_dir=ANS_DIR, shift_check=True, compare_full=True, name="test", ex1_uses_emb=False):
    # ex1_uses_emb=False: Exercise 1 and 2 are the NO-EMBEDDING forecast, so that Exercise 3 (images, images+RNAseq) is compared with them on the same patients.
    model = FINAL if model is None else model; sr = SR_MODEL if sr is None else sr; rep = REPORT if rep is None else rep; st = f"H handler:{name}"
    need = ["demographics.csv", "patient_assignment.csv", "treatments.csv", "surgeries.csv"]
    missing = [f for f in need if not url_exists(test_dir, f)]
    if not url_exists(test_dir, "visits.csv"): missing += [f for f in ["visits1.csv", "visits2.csv"] if not url_exists(test_dir, f)]
    assert not missing, f"missing files in {test_dir}: {missing}"
    C1, C2 = ["patient_id", "pred_avg_ihs4_next6"], ["patient_id", "pred_avg_ihs4_next6", "super_responder_prob"]
    TP = {k: (read_template(p_, c_) if p_ else None, c_) for k, (p_, c_) in dict(ex1=(ex1, C1), ex2=(ex2, C2), ex3a=(ex3a, C1), ex3b=(ex3b, C1)).items()}
    NICE = dict(ex1="Exercise 1", ex2="Exercise 2", ex3a="Exercise 3a (images)", ex3b="Exercise 3b (images + RNAseq)")
    # -- H1 the templates themselves
    ID = {}
    for k, (t_, c_) in TP.items():
        if t_ is None: ID[k] = pd.Index([], dtype=object); continue
        check(st + " H1 template", f"{NICE[k]} has the columns " + "/".join(c_) + " in that order", list(t_.columns) == c_, str(list(t_.columns)))
        check(st + " H1 template", f"{NICE[k]} patient_id unique and not empty", t_.patient_id.is_unique and t_.patient_id.notna().all(), f"{len(t_)} rows"); ID[k] = pd.Index(t_.patient_id)
    check(st + " H1 template", "Exercise 2 ids are a subset of Exercise 1 ids", ID["ex2"].isin(ID["ex1"]).all(), f"{int((~ID['ex2'].isin(ID['ex1'])).sum())} outside", warn=True)
    check(st + " H1 template", "Exercise 3b ids are a subset of Exercise 3a ids (everyone with RNAseq also has an image)", ID["ex3b"].isin(ID["ex3a"]).all(), f"{int((~ID['ex3b'].isin(ID['ex3a'])).sum())} outside", warn=True)
    ids_all = pd.Index(list(dict.fromkeys(sum([list(ID[k]) for k in ("ex1", "ex2", "ex3a", "ex3b")], []))), name="patient_id")
    # -- H2 the test tables (same integrity checks as train) and the link between templates and tables
    Tt = load_tables(test_dir); check_tables(Tt, step=st + " H2 tables")
    try: emb_t = load_emb(test_dir)
    except Exception as e: print("embeddings not loaded:", repr(e)); emb_t = {}
    print({k: len(v) for k, v in emb_t.items()} or "no embedding files found -> forecast core only")
    in_dem = ids_all.isin(set(Tt["dem"].patient_id)); n_test = Tt["dem"].patient_id.nunique()
    check(st + " H2 tables", "every template patient_id exists in the test demographics", in_dem.all(), f"{int((~in_dem).sum())} of {len(ids_all)} not found" + (f" e.g. {list(ids_all[~in_dem][:3])}" if (~in_dem).any() else ""))
    img_ids, rna_ids = set(emb_t.get("images", pd.DataFrame({"patient_id": []})).patient_id), set(emb_t.get("rnaseq", pd.DataFrame({"patient_id": []})).patient_id)
    check(st + " H2 tables", "every Exercise 3a id has an image embedding", all(i in img_ids for i in ID["ex3a"]), f"{sum(i not in img_ids for i in ID['ex3a'])} without", warn=True)
    check(st + " H2 tables", "every Exercise 3b id has an image AND an RNAseq embedding", all((i in img_ids and i in rna_ids) for i in ID["ex3b"]), f"{sum(not (i in img_ids and i in rna_ids) for i in ID['ex3b'])} without", warn=True)
    print(f"template ids: " + " | ".join(f"{k} {len(v)}" for k, v in ID.items()) + f" | union {len(ids_all)} | patients in test tables {n_test} -> predicting {len(ids_all)} of them ({len(ids_all) / n_test:.0%})")
    # -- H3 predict ONLY the listed ids
    scen = predict_scenarios(model, Tt, ids_all, emb_t, rep=rep); f = scen["f"]
    if shift_check:
        auc_, dt_ = shift_report(model.ref, f.reindex(columns=model.cols), model.cols, name=name); print(dt_.round(3).to_string())
        check(st + " H3 shift", "adversarial AUC train vs the listed patients < 0.75 (0.5 = same population)", auc_ < 0.75, f"AUC {auc_:.3f}", warn=True)
    if compare_full and n_test > len(ids_all):
        rest = pd.Index(sorted(set(Tt["dem"].patient_id) - set(ids_all))); f_rest = build(Tt, END, uplift=model.uplift, ids=rest)
        a = adversarial_validation(f.reindex(columns=model.cols), f_rest.reindex(columns=model.cols), model.cols)
        check(st + " H3 shift", "listed patients look like the rest of the test set (adversarial AUC < 0.65)", a < 0.65, f"AUC {a:.3f} on {len(ids_all)} listed vs {len(rest)} not listed", warn=True)
    P1 = scen["final"] if ex1_uses_emb else scen["M0"]
    out = {}
    for k, pred_ in dict(ex1=P1, ex2=P1, ex3a=scen["M0+images"], ex3b=scen["M0+rnaseq"]).items():
        t_ = TP[k][0]
        if t_ is None: out[k] = None; continue
        a_ = t_.copy(); a_["pred_avg_ihs4_next6"] = pred_.reindex(a_.patient_id).values; out[k] = a_
    # -- super-responder probability (every biologic patient in the list; Exercise 2 gets its rows)
    bio_ids = ids_all[ids_all.isin(cur_lines(Tt).index)]; S = sr.score(Tt, f, emb=emb_t, ids=bio_ids).set_index("patient_id") if len(bio_ids) else pd.DataFrame(columns=["super_responder_prob", "sr_source"])
    if out["ex2"] is not None:
        S2 = sr.score(Tt, f, emb=emb_t, ids=ID["ex2"]).set_index("patient_id"); out["ex2"]["super_responder_prob"] = S2.super_responder_prob.reindex(out["ex2"].patient_id).values
    # -- H4 predictions
    ymax = float(y.max())
    for k in ("ex1", "ex2", "ex3a", "ex3b"):
        a_ = out[k]
        if a_ is None: continue
        t_ = TP[k][0]; v = a_.pred_avg_ihs4_next6
        check(st + " H4 predictions", f"{NICE[k]}: one prediction per template row, same order, no NaN", len(a_) == len(t_) and (a_.patient_id.values == t_.patient_id.values).all() and v.notna().all(), f"{len(a_)} rows, {int(v.isna().sum())} NaN")
        check(st + " H4 predictions", f"{NICE[k]}: predictions are >= 0 and <= the largest train target ({ymax:.0f})", bool(v.between(0, ymax).all()), f"min {v.min():.2f} max {v.max():.2f}")
    mtr = float(y.mean()); m1 = float(out["ex1"].pred_avg_ihs4_next6.mean())
    check(st + " H4 predictions", "Exercise 1 mean prediction is within 50% of the train mean target (population check, not accuracy)", abs(m1 / mtr - 1) < 0.5, f"{m1:.2f} vs train {mtr:.2f}", warn=True)
    hq = pd.DataFrame({"h": f.hurley.reindex(out["ex1"].patient_id).values, "p": out["ex1"].pred_avg_ihs4_next6.values}).groupby("h").p.agg(["median", "size"]); tq = y.groupby(X.hurley).median()
    bad = [h for h in hq.index if h in tq.index and not (0.5 < hq.loc[h, "median"] / tq[h] < 2)]
    check(st + " H4 predictions", "median prediction per Hurley stage is within a factor 2 of the train median", not bad, "; ".join(f"H{int(h)}: {hq.loc[h,'median']:.1f} (n={int(hq.loc[h,'size'])}) vs train {tq.get(h, np.nan):.1f}" for h in hq.index), warn=True)
    if out["ex2"] is not None:
        d_ = (out["ex2"].set_index("patient_id").pred_avg_ihs4_next6 - out["ex1"].set_index("patient_id").pred_avg_ihs4_next6.reindex(out["ex2"].patient_id)).abs().max()
        check(st + " H4 predictions", "Exercise 2 forecast equals the Exercise 1 forecast for the same patient", d_ < 1e-9, f"max abs diff {d_:.2e}")
    for k in ("ex3a", "ex3b"):
        if out[k] is None: continue
        d3 = (out[k].set_index("patient_id").pred_avg_ihs4_next6 - scen["M0"].reindex(out[k].patient_id).values).abs(); nch = int((d3 > 1e-9).sum())
        check(st + " H4 predictions", f"{NICE[k]}: the embedding changes the forecast only where it was validated (recent biologic starters) and by a bounded amount", d3.max() < 5, f"{nch} of {len(d3)} patients changed ({nch / max(len(d3), 1):.0%}); max change {d3.max():.2f}; mean change among changed {d3[d3 > 1e-9].mean() if nch else 0:.2f}")
    # -- H5 super-responder
    if out["ex2"] is not None:
        s_ = out["ex2"].super_responder_prob; on_b = ID["ex2"].isin(cur_lines(Tt).index); sro = S2.sr_source.reindex(out["ex2"].patient_id)
        check(st + " H5 super-responder", "probabilities are in [0, 1], no NaN, one per Exercise 2 row", bool(s_.notna().all() and s_.between(0, 1).all()), f"min {s_.min():.3f} max {s_.max():.3f}")
        check(st + " H5 super-responder", "Exercise 2 patients are on a biologic at 31 Dec 2025 (the detector only means something for them)", on_b.mean() > 0.95, f"{on_b.mean():.1%} on a biologic; {int((~on_b).sum())} filled with the train base rate", warn=True)
        check(st + " H5 super-responder", "mean probability is near the train base rate (within a factor 2)", 0.5 < s_.mean() / sr.rate < 2, f"mean {s_.mean():.3f} vs train rate {sr.rate:.3f}", warn=True)
        rho = out["ex2"].super_responder_prob.corr(out["ex2"].pred_avg_ihs4_next6, method="spearman")
        check(st + " H5 super-responder", "probability goes down when the predicted IHS4 goes up (Spearman < -0.2)", rho < -0.2, f"Spearman {rho:.2f}", warn=True)
        print(f"super-responder prob: mean {s_.mean():.3f} | >0.5: {(s_ > 0.5).sum()} | source: {sro.value_counts().to_dict()}")
    # -- patient report (everything else the clinician asked for), for every listed patient
    R = rep.make(Tt, f, ids_all, scen, emb_t, sr_prob=S.super_responder_prob if len(S) else None)
    for k, col in dict(ex1="in_exercise1", ex2="in_exercise2", ex3a="in_exercise3a_images", ex3b="in_exercise3b_images_rnaseq").items(): R[col] = R.patient_id.isin(ID[k]).astype(int)
    R["pred_no_embeddings"] = scen["M0"].reindex(R.patient_id).values; R["pred_with_images"] = scen["M0+images"].reindex(R.patient_id).values; R["pred_with_rnaseq"] = scen["M0+rnaseq"].reindex(R.patient_id).values
    R["pred_avg_ihs4_next6"] = P1.reindex(R.patient_id).values
    R = R.round(4)
    hr = f"{st} H7 report"
    check(hr, "one row per listed patient, same order", len(R) == len(ids_all) and (R.patient_id.values == ids_all.values).all(), f"{len(R)} rows")
    check(hr, "age is plausible (5-100) and sex is present", bool(R.age_years.between(5, 100).all() and R.sex.notna().all()), f"age {R.age_years.min():.0f}-{R.age_years.max():.0f}")
    ls = R.last_score_age_days.dropna(); check(hr, "last score is never in the future and its age is >= 0", bool((ls >= 0).all()), f"{len(ls)} patients with a score; median age {ls.median():.0f} d; {R.last_ihs4_score.isna().sum()} without any score")
    check(hr, "current treatment text exists for everyone and matches the biologic flag", bool(R.current_treatment.notna().all() and (~R.current_treatment.str.startswith("no biologic") == (R.on_biologic == 1)).all()), f"{int(R.on_biologic.sum())} on a biologic")
    ev = rep.ev["worsen"]; fs = R.worsen_flag.mean()
    check(hr, "p_worsen is a probability; the share flagged is within a factor 2 of train out-of-fold", bool(R.p_worsen.between(0, 1).all() and 0.5 * ev["flag_share"] < fs < 2 * ev["flag_share"]), f"flagged {fs:.1%} vs train {ev['flag_share']:.1%}; mean p {R.p_worsen.mean():.3f} vs train {ev['mean_oof']:.3f}", warn=True)
    check(hr, "flagged patients have a higher expected change than unflagged ones", bool(R.loc[R.worsen_flag == 1, "expected_change"].mean() > R.loc[R.worsen_flag == 0, "expected_change"].mean()) if R.worsen_flag.nunique() > 1 else True, f"flagged {R.loc[R.worsen_flag==1,'expected_change'].mean():+.2f} vs {R.loc[R.worsen_flag==0,'expected_change'].mean():+.2f}")
    rb = R[R.on_biologic == 1]
    check(hr, "every biologic patient has a response type; probabilities add to 1", bool(rb.response_type.notna().all() and np.allclose(rb[["p_deep", "p_partial", "p_minimal"]].sum(1), 1, atol=1e-3)) if len(rb) else True, f"{len(rb)} biologic; sources {rb.response_type_source.str.split(' ').str[0].value_counts().to_dict() if len(rb) else {}}")
    if len(rb): check(hr, "average response-type probabilities are within 15 points of the train shares", bool(all(abs(rb["p_" + t].mean() - rep.ev['resp']['shares'][t]) < 0.15 for t in rep.names)), f"test mean p {({t: round(float(rb['p_' + t].mean()), 2) for t in rep.names})} vs train {({t: float(v) for t, v in rep.ev['resp']['shares'].items()})}; confident (>0.6) calls: {(rb.response_confidence > 0.6).mean():.0%}", warn=True)
    check(hr, "a top risk factor (and a top clinical factor) is named for everyone", bool(R.top_risk_factor.notna().all() and R.top_clinical_factor.notna().all()), f"most common: {R.top_risk_factor.value_counts().head(3).to_dict()}")
    h23 = R.hurley_stage.isin([2, 3]); sp = R.loc[h23, "p_surgery_benefit"].dropna()
    check(hr, "surgery probabilities exist only for Hurley 2-3 and are in [0, 1]", bool(R.loc[~h23, "p_surgery_benefit"].isna().all() and sp.between(0, 1).all()), f"{len(sp)} of {int(h23.sum())} Hurley 2-3 patients have an estimate; mean benefit {sp.mean():.3f}")
    check(hr, "super_responder_prob is filled for every biologic patient and empty otherwise", bool(R.loc[R.on_biologic == 1, "super_responder_prob"].notna().all() and R.loc[R.on_biologic == 0, "super_responder_prob"].isna().all()), "")
    # -- write the filled templates + report and re-read them
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True); files = []
    for k, ex_ in dict(ex1=ex1, ex2=ex2, ex3a=ex3a, ex3b=ex3b).items():
        a_ = out[k]
        if a_ is None: continue
        fp = out_dir / os.path.basename(str(ex_).split("?")[0]); a_.to_csv(fp, index=False); files.append(fp)
        b_ = pd.read_csv(fp, dtype={"patient_id": str}); check(st + " H6 files", f"{fp.name} re-read gives the same rows and columns", list(b_.columns) == list(a_.columns) and len(b_) == len(a_) and b_.iloc[:, 1:].notna().all().all() and np.allclose(b_.iloc[:, 1:].values, a_.iloc[:, 1:].values), f"{len(b_)} rows")
        print("written:", fp)
    fr = out_dir / f"patient_report_{name}.csv"; R.to_csv(fr, index=False); files.append(fr); b_ = pd.read_csv(fr); check(st + " H6 files", f"{fr.name} re-read gives the same rows and columns", len(b_) == len(R) and list(b_.columns) == list(R.columns), f"{len(b_)} rows x {b_.shape[1]} columns"); print("written:", fr)
    return dict(ex1=out["ex1"], ex2=out["ex2"], ex3a=out["ex3a"], ex3b=out["ex3b"], report=R, scen=scen, sr=(S.reset_index() if len(S) else None), files=files, features=f)


NameError: name 'ANS_DIR' is not defined

In [ ]:
# input/output test dataset 

TEST_SET    = "test2"        # <- "test1" or "test2"; this is the only line to change
SHIFT_CHECK = True

SETS = {"test1": dict(name="test_1", folder="Raw_data_test1"),
        "test2": dict(name="test_2", folder="Raw_data_test2")}
S         = SETS[TEST_SET]
TEST_NAME = S["name"]
GH        = "https://github.com/FairzanIsLearning/Chemometric-learn"
TEST_DIR  = f"{GH}/blob/main/Mixed/HS_Model/{S['folder']}"                                     # tables (your _src turns it into a raw link)
RAW_TPL   = f"https://raw.githubusercontent.com/FairzanIsLearning/Chemometric-learn/main/Mixed/HS_Model/{S['folder']}/"   # answer templates

# local folders: one per test set, so test1 and test2 never overwrite each other
LOCAL_DIR = Path("/content/HS_Model_local") / TEST_NAME
TPL_DIR   = LOCAL_DIR / "templates"
ANS_DIR   = LOCAL_DIR / "answers"
for d in (TPL_DIR, ANS_DIR): d.mkdir(parents=True, exist_ok=True)

# Google Drive: mount once, never delete anything
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Drive is not mounted"
DRIVE_DIR = Path("/content/drive/MyDrive/HS_Model/outputs") / TEST_NAME
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# kept for the older single-file cells further down
OUT_CSV = ANS_DIR / f"predictions_{TEST_NAME}.csv"
SUB_CSV = ANS_DIR / f"answers_teamName_exercise1_{TEST_NAME}.csv"

print("test set :", TEST_NAME, "|", TEST_DIR)
print("local    :", LOCAL_DIR)
print("drive    :", DRIVE_DIR)

In [ ]:
# ---- 1. get the blank templates for the chosen test set onto the local disk
TPL = {"answers_teamName_exercise1.csv": True,                 # name: required?
       "answers_teamName_exercise2.csv": False,
       "answers_teamName_exercise3_images.csv": False,
       "answers_teamName_exercise3_images_rnaseq.csv": False}

for name in TPL:
    dst = TPL_DIR / name
    if dst.exists(): continue
    try:
        urllib.request.urlretrieve(RAW_TPL + name, str(dst)); print("downloaded:", name)
    except Exception:
        print("not on GitHub for", TEST_NAME, ":", name)

if not (TPL_DIR / "answers_teamName_exercise1.csv").exists():      # last resort: upload by hand
    print(f"Select the {TEST_NAME} answer template(s) from your computer:")
    for fn, data in files.upload().items(): (TPL_DIR / fn).write_bytes(data)
assert (TPL_DIR / "answers_teamName_exercise1.csv").exists(), "exercise1 template still missing"

pick = lambda n: str(TPL_DIR / n) if (TPL_DIR / n).exists() else None
IDS_EX1  = pick("answers_teamName_exercise1.csv")
IDS_EX2  = pick("answers_teamName_exercise2.csv")
IDS_EX3A = pick("answers_teamName_exercise3_images.csv")
IDS_EX3B = pick("answers_teamName_exercise3_images_rnaseq.csv")
print("ex1:", bool(IDS_EX1), "| ex2:", bool(IDS_EX2), "| ex3a:", bool(IDS_EX3A), "| ex3b:", bool(IDS_EX3B))

# ---- 2. run: answers are written to the LOCAL folder first
res = run_template_handler(TEST_DIR, IDS_EX1, IDS_EX2, IDS_EX3A, IDS_EX3B,
                           out_dir=ANS_DIR, shift_check=SHIFT_CHECK, name=TEST_NAME)

print("\nwritten locally:")
for fp in res["files"]:
    print("OK" if Path(fp).exists() else "MISSING", "|", fp)
print("Exercise 1 rows:", len(res["ex1"]), "| NaN:", int(res["ex1"].pred_avg_ihs4_next6.isna().sum()))
if res["ex2"] is not None:
    print("Exercise 2 rows:", len(res["ex2"]), "| NaN:", int(res["ex2"][["pred_avg_ihs4_next6", "super_responder_prob"]].isna().sum().sum()))

# ---- 3. upload to Drive (copy only, then verify size and content)
import filecmp
print("\ncopy to Drive:", DRIVE_DIR)
bad = []
for fp in res["files"]:
    dst = DRIVE_DIR / Path(fp).name
    shutil.copy2(fp, dst)
    ok = dst.exists() and filecmp.cmp(fp, dst, shallow=False)
    print("OK     " if ok else "FAILED ", "|", dst.name, f"({dst.stat().st_size/1024:.0f} KB)")
    if not ok: bad.append(dst.name)
assert not bad, f"Drive copy failed for {bad}"
print("Drive folder now contains:", sorted(os.listdir(DRIVE_DIR)))

# ---- 4. optional: also download to your computer
# for fp in res["files"]: files.download(str(fp))

res["report"].head()

## 11. The validation ledger (one line per check) and what is left open

In [ ]:
LG = pd.DataFrame(LEDGER); print(f"{int((LG.status=='PASS').sum())} of {len(LG)} checks passed | {int((LG.status=='WARN').sum())} warning(s) | {int((LG.status=='FAIL').sum())} failed")
H_ = LG[LG.step.str.startswith("H handler")]; print(f"handler (template) checks: {int((H_.status=='PASS').sum())} PASS / {int((H_.status=='WARN').sum())} WARN / {int((H_.status=='FAIL').sum())} FAIL")
print("WARNINGS / FAILURES:\n", LG[LG.status != "PASS"][["step", "check", "status", "detail"]].to_string() if (LG.status != "PASS").any() else "none")
FIND.to_csv("findings_ledger.csv"); LG.to_csv("validation_ledger.csv", index=False)
cols = {"PASS": "#d9ead3", "WARN": "#fff2cc", "FAIL": "#f4cccc"}
LGv = pd.concat([H_, LG[~LG.step.str.startswith("H handler")]])      # handler rows first
rows = "".join(f"<tr><td>{r.step}</td><td>{r.check}</td><td style='background:{cols[r.status]};font-weight:bold'>{r.status}</td><td>{r.detail}</td></tr>" for r in LGv.itertuples())
HTMLTable("<style>table.l td,table.l th{border:1px solid #bbb;padding:4px;font-size:12px;text-align:left}</style><table class='l'><tr><th>step</th><th>check</th><th>result</th><th>detail</th></tr>" + rows + "</table>")
